# Aquatic Guardian — Mentee 2 (Workstream D, production side)
## Aquaculture Vision: Measurement, Biomass, Welfare and Bioenergetics under a Thermal-Reuse Loop

**Six-week Colab project. Free tier, open-source only, public data.**

---

### What this notebook is

The thermal notebook ends with a conditional finding: low-grade heat is only usable through
**capture at the outfall**, feeding a **contained** aquaculture loop. That finding is
worthless without an answer to the next question:

> *If we run that loop, are the fish actually growing, and are they actually all right?*

This notebook produces that evidence. It is a **measurement system**, and measurement systems
are judged on bias, precision and traceability — not on whether the masks look nice.

```
  thermal_handoff.json  ──▶  loop inlet temperature, DO, NH3 fraction
        (Notebook A)                     │
                                         ▼
  camera optics ──▶ rendered tank ──▶ instance segmentation ──▶ morphometrics
  (pinhole + flat-port     (exact GT:      (watershed splits      (curvilinear length,
   refraction, Jerlov       length, mass,   touching fish)         not chord)
   attenuation)             K, occlusion)        │                      │
                                                 ▼                      ▼
                                    size-selectivity bias   ──▶  biomass with
                                    correction                    bootstrap CI
                                                 │                      │
                                                 ▼                      ▼
                                    SWIM-style weighted        bioenergetics: TGC growth,
                                    welfare index              O2 demand, TAN load, NH3
                                                 │                      │
                                                 └──────┬───────────────┘
                                                        ▼
                                     stocking-density ceiling + reuse-decision evidence
```

### The measurement chain, and where it breaks

Every stage multiplies error into the final biomass number, because mass goes as $L^{b}$
with $b \approx 3$:

| Stage | Error source | Effect on mass |
|---|---|---|
| range estimate | ±10 % in $Z$ | **±33 %** |
| flat-port refraction ignored | $n_w = 1.34$ | **+34 % in length → +140 % in mass** |
| bent fish measured chord-wise | 60° body bend | **−13 %** |
| touching fish merged | one blob, two fish | count halved, sizes doubled |
| log-normal residual not corrected | Jensen's inequality | few % but systematic |
| size-selective visibility | small fish missed | biomass **over**-estimated |

Each of these is derived, implemented and **measured against exact ground truth** below. The
point of the notebook is that a 3 % camera-calibration slip is a 9 % biomass error, and a
biomass error is a stocking-density error, and a stocking-density error is a welfare problem.

### How this differs from the other two notebooks

| | Mentee 1 | **This notebook** | Thermal notebook |
|---|---|---|---|
| Where | receiving water, outside | **inside the contained loop** | plume + plant |
| Task | detect / classify / alert | **segment and measure** | transport + compliance |
| Scored on | AP, calibration, alert precision | **length/mass bias and CI, welfare agreement** | analytic benchmarks |

### Scope reminder

Every measurement is an **estimate with error**, and this notebook reports the error. Welfare
flags are **triage for human review, never diagnoses**. Condition-factor bands, welfare
weights and stocking limits are **placeholders to be set with aquatic-science and veterinary
input**. No welfare or thermal-reuse decision is made by a mentee alone.

---

### Six-week map

| Week | Parts | Deliverable |
|---|---|---|
| 1 | 0–2 | Dataset + licence; optics and calibration chain with error propagation |
| 2 | 3–4 | Tank renderer with exact GT; segmentation **passing its vetting suite** |
| 3 | 5–6 | Morphometrics (curvilinear length); biomass with bias corrections and CIs |
| 4 | 7–8 | Welfare index; tracking for individual growth and de-duplicated counts |
| 5 | 9–10 | Bioenergetics coupled to the thermal handoff; stocking-density ceiling |
| 6 | 11–12 | Evaluation, power analysis, safety rules, reproducible handoff |

## Part 0 — Environment, run configuration, and the safety contract

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time, os
from scipy import ndimage as ndi, stats
from scipy.optimize import linear_sum_assignment, minimize_scalar

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.width", 165); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9, "figure.facecolor": "white"})

RUN = dict(SEED=20260219, RUN_REAL_DATA=False, FAST=True, EXPORT_DIR=".")
RNG = np.random.default_rng(RUN["SEED"])

SAFETY_CONTRACT = {
  "role": "measurement and triage evidence for human review",
  "never": ["diagnose disease", "authorise a thermal-reuse decision",
            "report biomass without an uncertainty interval",
            "treat a welfare flag as a finding"],
  "thresholds_are": "placeholders pending aquatic-science and veterinary input",
  "human_review_required_for": ["every welfare flag", "any stocking-density change",
                                "any claim that the loop is operating safely"],
  "containment": "contained loop only. No release of cultured organisms to the wild, ever.",
}
print("Aquatic Guardian — Mentee 2 — aquaculture vision and bioenergetics")
for k, v in SAFETY_CONTRACT.items(): print(f"  {k}: {v}")

# --- couple to the thermal notebook -----------------------------------------
THERMAL = None
for pth in ["thermal_handoff.json", "./thermal_handoff.json", "/content/thermal_handoff.json"]:
    if os.path.exists(pth):
        THERMAL = json.load(open(pth)); print(f"\nLoaded thermal handoff from {pth}"); break
if THERMAL is None:
    THERMAL = dict(aquaculture_handoff=dict(loop_inlet_T_C=19.5, ambient_T_C=11.5,
                                            salinity=30.5, pH=7.95,
                                            DO_at_loop_T_mgL=7.63,
                                            NH3_fraction_at_loop_T=0.0335,
                                            captured_heat_MW=0.0),
                   key_results=dict(dT_nearfield_K=0.99))
    print("\nthermal_handoff.json not found — using documented stand-in values.")
    print("Run the thermal notebook first to couple the two models for real.")
AQ = THERMAL["aquaculture_handoff"]
print(json.dumps(AQ, indent=2))

## Part 1 — Datasets, licences, and what "ground truth" means here

Unlike a detection dataset, a **measurement** dataset needs paired physical truth: a fish
whose length and mass were recorded on a board and a scale. Very few public datasets have it,
which is why the renderer in Part 3 exists.

In [ ]:
DATASETS = pd.DataFrame([
 ("DeepFish (Garcia-d'Urso et al. 2022, Sci. Data)", "sea-cage aquaculture",
  "instance masks + SIZE labels", "open (Scientific Data)", "doi.org/10.1038/s41597-022-01416-0"),
 ("DeepFish-Seg / FishSeg (Saleh et al. 2020)", "reef, ~620 masks",
  "segmentation only", "MIT / CC BY 4.0", "github.com/alzayats/DeepFish"),
 ("Large-Scale Fish Dataset (Ulucan et al. 2020)", "9 seafood species, RGB + masks",
  "segmentation + classification, NO in-situ size", "research use; check the Kaggle page",
  "kaggle.com/datasets/crowww/a-large-scale-fish-dataset"),
 ("Fish4Knowledge", "reef video, 10 years", "detection/tracking, no size",
  "check per-clip terms", "homepages.inf.ed.ac.uk/rbf/Fish4Knowledge"),
], columns=["dataset", "content", "suitability for MEASUREMENT", "licence", "source"])
display(DATASETS)
print("Only the first carries size labels. For everything else you can validate SEGMENTATION")
print("but not MEASUREMENT, and conflating the two is the standard way an aquaculture-vision")
print("paper overstates its accuracy. If you use a mask-only dataset, say explicitly that the")
print("length error is UNVALIDATED on real data.")
print("\nGround truth for measurement needs, per fish: mask, fork length from a measuring board,")
print("wet mass from a scale, and the camera-to-fish range. Budget a handling session with a")
print("veterinary/welfare protocol; it is the only way to close the loop.")

## Part 2 — Optics: the calibration chain and how its errors propagate

### 2.1 Pinhole projection and the flat-port trap

For a pinhole camera of focal length $f$ (px) viewing an object of length $L$ at range $Z$:

$$L_{px} = \frac{f\,L}{Z}$$

Underwater, rays refract at the **flat port** between water ($n_w \approx 1.34$) and the air
behind the glass. In the paraxial limit this is equivalent to an effective focal length
$f_{\text{eff}} = n_w f$:

$$L_{px} = \frac{n_w f\,L}{Z}\quad\Longrightarrow\quad
 L = \frac{L_{px}\,Z}{n_w f}$$

**Ignore the $n_w$ and every fish is 34 % too long**, which is $1.34^3 = 2.4\times$ too heavy.
This is the single largest error in the chain and it is a one-line fix.

### 2.2 Error propagation to mass

With $W = aL^{b}$, relative errors compose multiplicatively:

$$\frac{\sigma_W}{W} = b\,\frac{\sigma_L}{L}
 = b\sqrt{\left(\frac{\sigma_{L_{px}}}{L_{px}}\right)^2
 + \left(\frac{\sigma_Z}{Z}\right)^2 + \left(\frac{\sigma_f}{f}\right)^2}$$

Because $b \approx 3$, **every relative error is tripled**. Range is usually the weak link:
a monocular camera has no range at all and must infer it from apparent size, which is
circular. Stereo or a structured reference in the frame is not optional for real work.

In [ ]:
N_WATER = 1.34

def px_per_cm(f_px, Z_cm, n_w=N_WATER):
    '''Effective scale behind a flat port (paraxial).'''
    return n_w*f_px/Z_cm

def length_from_px(L_px, Z_cm, f_px, n_w=N_WATER):
    return L_px*Z_cm/(n_w*f_px)

def stereo_range(disparity_px, baseline_cm, f_px, n_w=N_WATER):
    '''Range from stereo disparity. The refractive index enters through f_eff.'''
    return n_w*f_px*baseline_cm/np.maximum(disparity_px, 1e-9)

CAM = dict(f_px=1400.0, baseline_cm=12.0, sensor_noise_px=0.6, f_cal_rel_err=0.02)

rows = []
for Z in [60, 100, 150, 250, 400]:
    s = px_per_cm(CAM["f_px"], Z)
    d = CAM["f_px"]*N_WATER*CAM["baseline_cm"]/Z
    sZ_rel = CAM["sensor_noise_px"]/d          # dZ/Z = d(disp)/disp for stereo
    rows.append(dict(Z_cm=Z, px_per_cm=round(s, 2), px_for_30cm_fish=round(30*s),
                     disparity_px=round(d, 1), range_rel_err_pct=round(100*sZ_rel, 2)))
OPT = pd.DataFrame(rows).set_index("Z_cm")
display(OPT.style.set_caption(f"Stereo geometry, f={CAM['f_px']:.0f} px, "
                              f"baseline {CAM['baseline_cm']:.0f} cm"))

# ---- VET: refraction correction and the cost of skipping it ----------------
_Z, _L = 120.0, 32.0
_Lpx = px_per_cm(CAM["f_px"], _Z)*_L
_ok = length_from_px(_Lpx, _Z, CAM["f_px"])
_bad = _Lpx*_Z/CAM["f_px"]                       # refraction ignored
assert abs(_ok-_L) < 1e-9, "refraction-corrected inverse is not exact"
print(f"[VET] a {_L} cm fish at {_Z} cm spans {_Lpx:.0f} px; corrected inverse recovers "
      f"{_ok:.6f} cm -> PASS")
print(f"      ignoring the flat port gives {_bad:.2f} cm: {100*(_bad/_L-1):+.1f}% in length, "
      f"{100*((_bad/_L)**3-1):+.0f}% in MASS")

def mass_error_budget(rel_px, rel_Z, rel_f, b=3.0):
    rel_L = np.sqrt(rel_px**2 + rel_Z**2 + rel_f**2)
    return rel_L, b*rel_L

rows = []
for Z in [60, 100, 150, 250, 400]:
    d = CAM["f_px"]*N_WATER*CAM["baseline_cm"]/Z
    rel_Z = CAM["sensor_noise_px"]/d
    rel_px = 2*CAM["sensor_noise_px"]/(px_per_cm(CAM["f_px"], Z)*28.0)   # 2 endpoints, 28 cm fish
    rL, rW = mass_error_budget(rel_px, rel_Z, CAM["f_cal_rel_err"])
    rows.append(dict(Z_cm=Z, endpoint_err_pct=100*rel_px, range_err_pct=100*rel_Z,
                     focal_cal_err_pct=100*CAM["f_cal_rel_err"],
                     length_err_pct=100*rL, MASS_err_pct=100*rW))
BUD = pd.DataFrame(rows).set_index("Z_cm").round(2)
display(BUD.style.background_gradient(subset=["MASS_err_pct"], cmap="Reds")
        .set_caption("Per-fish error budget. Mass error is 3x length error, always."))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ZZ = np.linspace(40, 450, 200)
for nm, rel in [("endpoint localisation", 2*CAM["sensor_noise_px"]/(px_per_cm(CAM["f_px"], ZZ)*28.0)),
                ("stereo range", CAM["sensor_noise_px"]/(CAM["f_px"]*N_WATER*CAM["baseline_cm"]/ZZ)),
                ("focal calibration", np.full_like(ZZ, CAM["f_cal_rel_err"]))]:
    ax[0].plot(ZZ, 100*rel, label=nm)
tot = np.sqrt((2*CAM["sensor_noise_px"]/(px_per_cm(CAM["f_px"], ZZ)*28.0))**2
              + (CAM["sensor_noise_px"]/(CAM["f_px"]*N_WATER*CAM["baseline_cm"]/ZZ))**2
              + CAM["f_cal_rel_err"]**2)
ax[0].plot(ZZ, 100*tot, "k", lw=1.8, label="combined length error")
ax[0].set_xlabel("range Z (cm)"); ax[0].set_ylabel("relative error (%)")
ax[0].legend(fontsize=6.5); ax[0].set_title("Which term dominates, and where")
ax[1].plot(ZZ, 300*tot, color="#C0392B", lw=1.8)
ax[1].axhline(10, color="k", ls="--", lw=.8); ax[1].annotate("10% mass error", (60, 11), fontsize=7)
ax[1].set_xlabel("range Z (cm)"); ax[1].set_ylabel("per-fish MASS error (%)")
ax[1].set_title("The b=3 amplification")
bias = np.linspace(-0.06, 0.06, 100)
ax[2].plot(100*bias, 100*((1+bias)**3-1), color="#1F3A5F", lw=1.8)
ax[2].axvline(2, color="crimson", ls=":"); ax[2].axhline(6.1, color="crimson", ls=":")
ax[2].annotate("a 2% calibration slip\n= 6% biomass bias", (-5.5, 8), fontsize=7)
ax[2].set_xlabel("systematic length bias (%)"); ax[2].set_ylabel("systematic biomass bias (%)")
ax[2].set_title("Bias does not average out")
plt.tight_layout(); plt.show()
print("Random error shrinks as 1/sqrt(n) across the sample. SYSTEMATIC error does not.")
print("A mis-measured focal length biases every fish in the same direction forever, so the")
print("calibration target and its date belong in the repo next to the model weights.")

## Part 3 — A tank renderer with exact ground truth

Each fish is drawn from an **articulated spine** with a controllable body bend, projected
through the Part 2 optics at a known range, composited with correct depth ordering (so
occlusion is exact, not approximated), and degraded by Jerlov-style attenuation and
backscatter. Every fish carries its true length, mass, condition factor, bend angle,
occlusion fraction and range.

That is what lets Part 5 report **bias** rather than "it looked about right".

In [ ]:
H_IMG, W_IMG = 300, 460
SENSOR_NOISE = 0.010
TANK = dict(bD=[0.35, 0.12, 0.16], bB=[0.28, 0.11, 0.14], Binf=[0.10, 0.30, 0.34])

def attenuate(I, z_m):
    bD, bB, Binf = (np.array(TANK[k]) for k in ("bD", "bB", "Binf"))
    return I*np.exp(-bD*z_m) + Binf*(1-np.exp(-bB*z_m))

# --- length-weight and condition factor -------------------------------------
LW = dict(a=0.0115, b=3.02, sigma_lnW=0.11,
          source="illustrative salmonid-like values; look up per species/life stage on FishBase")
def mass_from_length(L_cm, a=LW["a"], b=LW["b"]): return a*np.asarray(L_cm, float)**b
def fulton_K(W_g, L_cm): return 100.0*np.asarray(W_g, float)/np.asarray(L_cm, float)**3

def _spine(L_px, bend_rad, n=90):
    if abs(bend_rad) < 1e-6:
        t = np.linspace(-L_px/2, L_px/2, n); return t, np.zeros(n)
    R = L_px/bend_rad
    th = np.linspace(-bend_rad/2, bend_rad/2, n)
    return R*np.sin(th), R*(1-np.cos(th))

def draw_fish(canvas, zbuf, cx, cy, L_px, ang, bend, depth_m, radiance, girth_ratio=0.22):
    h, w = canvas.shape[:2]
    sx, sy = _spine(L_px, bend)
    ca, sa = np.cos(ang), np.sin(ang)
    px = cx + sx*ca - sy*sa; py = cy + sx*sa + sy*ca
    yy, xx = np.mgrid[0:h, 0:w]
    body = np.zeros((h, w), bool)
    u = np.linspace(-1, 1, len(sx))
    for X, Yc, uu in zip(px, py, u):
        r = L_px*girth_ratio*np.sqrt(max(1e-6, 1-0.88*uu**2))
        if uu < -0.86:                       # caudal fin flare
            r = L_px*girth_ratio*0.95*abs(uu+0.86)/0.14 + L_px*0.012
        body |= ((xx-X)**2 + (yy-Yc)**2) <= r*r
    vis = body & (depth_m < zbuf)
    canvas[vis] = radiance
    zbuf[vis] = depth_m
    return body, vis, (px, py)

def render_tank(n_fish=14, L_mean=27.0, L_sd=4.5, bend_max_deg=55, deform_frac=0.12,
                Z_range=(90, 220), seed=None, f_px=CAM["f_px"], turbid=1.0):
    r = np.random.default_rng(seed) if seed is not None else RNG
    yy, xx = np.mgrid[0:H_IMG, 0:W_IMG]/max(H_IMG, W_IMG)
    canvas = np.stack([0.16+0.05*xx, 0.27+0.07*yy, 0.30+0.05*(1-yy)], -1)
    canvas = np.clip(canvas + r.normal(0, .008, canvas.shape), 0, 1)
    canvas = attenuate(canvas, 2.2*turbid)
    zbuf = np.full((H_IMG, W_IMG), 1e9)
    truth, order = [], []
    for _ in range(n_fish):
        Z = float(r.uniform(*Z_range))
        L_cm = float(np.clip(r.normal(L_mean, L_sd), 8, None))
        order.append((Z, L_cm))
    for Z, L_cm in sorted(order, key=lambda t: -t[0]):        # far to near
        scale = px_per_cm(f_px, Z)
        L_px = L_cm*scale
        if L_px < 14 or L_px > 0.95*W_IMG: continue
        cx, cy = r.uniform(.10*W_IMG, .90*W_IMG), r.uniform(.15*H_IMG, .85*H_IMG)
        ang = r.uniform(0, 2*np.pi)
        bend = np.deg2rad(r.uniform(-bend_max_deg, bend_max_deg))
        deformed = r.random() < deform_frac
        girth = r.uniform(.28, .34) if deformed else r.uniform(.19, .24)
        W_true = mass_from_length(L_cm)*np.exp(r.normal(0, LW["sigma_lnW"]))
        if deformed: W_true *= r.uniform(.62, .80)            # emaciated / deformed
        I_obj = np.array([r.uniform(.55, .78), r.uniform(.50, .70), r.uniform(.30, .46)])
        rad = attenuate(I_obj, Z/100.0*turbid)
        body, vis, _ = draw_fish(canvas, zbuf, cx, cy, L_px, ang, bend, Z, rad, girth)
        if vis.sum() < 60: continue
        ys, xs = np.nonzero(vis)
        truth.append(dict(L_cm=L_cm, mass_g=W_true, K=float(fulton_K(W_true, L_cm)),
                          Z_cm=Z, L_px_true=L_px, bend_deg=float(np.rad2deg(abs(bend))),
                          girth_ratio=girth, deformed=bool(deformed),
                          occlusion=float(1-vis.sum()/max(body.sum(), 1)),
                          cx=float(xs.mean()), cy=float(ys.mean()),
                          area_px=int(vis.sum()), mask=vis))
    canvas = np.clip(canvas + r.normal(0, SENSOR_NOISE, canvas.shape), 0, 1)
    return canvas, truth

t0 = time.time()
IMG, TRUTH = render_tank(n_fish=16, seed=RUN["SEED"])
print(f"rendered {len(TRUTH)} visible fish in {time.time()-t0:.2f}s")
T0 = pd.DataFrame([{k: v for k, v in t.items() if k != "mask"} for t in TRUTH])
display(T0.describe().loc[["mean", "std", "min", "max"]].round(2)
        [["L_cm", "mass_g", "K", "Z_cm", "bend_deg", "occlusion"]])

# ---- VET: the renderer's own geometry is self-consistent -------------------
_err = max(abs(t["L_px_true"] - t["L_cm"]*px_per_cm(CAM["f_px"], t["Z_cm"])) for t in TRUTH)
assert _err < 1e-9, "renderer projection does not match the optics model"
print(f"[VET] rendered pixel lengths match f_eff*L/Z to {_err:.1e} px -> PASS")
_m = np.array([t["mass_g"] for t in TRUTH]); _l = np.array([t["L_cm"] for t in TRUTH])
print(f"[VET] rendered ln-mass residual sd = {np.std(np.log(_m/mass_from_length(_l))):.3f} "
      f"(target {LW['sigma_lnW']:.3f}; small n, so expect scatter)")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].imshow(IMG); ax[0].set_title(f"rendered tank, {len(TRUTH)} fish"); ax[0].axis("off")
ov = IMG.copy()
for t in TRUTH:
    e = ndi.binary_dilation(t["mask"], np.ones((3, 3))) & ~t["mask"]
    ov[e] = [1, 1, 0]
ax[1].imshow(ov); ax[1].set_title("exact ground-truth masks"); ax[1].axis("off")
zmap = np.full((H_IMG, W_IMG), np.nan)
for t in TRUTH: zmap[t["mask"]] = t["Z_cm"]
im = ax[2].imshow(zmap, cmap="viridis"); plt.colorbar(im, ax=ax[2], label="range (cm)")
ax[2].set_title("true range per fish (depth ordering is exact)"); ax[2].axis("off")
plt.tight_layout(); plt.show()
print(f"bend angles span {T0.bend_deg.min():.0f}-{T0.bend_deg.max():.0f} deg and occlusion")
print(f"reaches {100*T0.occlusion.max():.0f}%. Both are deliberate: they are the two failure")
print("modes that a measurement pipeline must be tested against, and neither is labelled in")
print("any public dataset.")

## Part 4 — Instance segmentation: splitting fish that touch

### 4.1 Why connected components is not enough

Background subtraction gives a foreground mask. Connected-component labelling then treats
**any two touching fish as one animal**, which halves the count and roughly doubles the
apparent length — an error that propagates as $2^3 = 8\times$ into that individual's mass.
Touching fish are not an edge case in a stocked tank; they are the normal condition.

### 4.2 Distance-transform watershed

The fix is classical and needs no GPU. Compute the Euclidean distance transform of the
foreground; each fish contributes a ridge with an interior maximum. Seed from the
**non-maximum-suppressed** local maxima, then flood the mask in order of **decreasing**
distance so basins grow from the inside out and meet at the narrow neck between two fish.

Two parameters matter and both are physical, not magic:

- `min_sep` — seeds closer than this are the same animal. Set it from the expected body
  half-width in pixels, which Part 2 gives you from range.
- `peak_frac` — a maximum below this fraction of the largest is a bump on a body, not a fish.

We verify on geometry with known answers before using it on anything.

In [ ]:
def foreground(img, k_open=2):
    corners = np.concatenate([img[:8, :8].reshape(-1, 3), img[:8, -8:].reshape(-1, 3),
                              img[-8:, :8].reshape(-1, 3), img[-8:, -8:].reshape(-1, 3)])
    bg = np.median(corners, 0)
    d = np.sqrt(((img-bg)**2).sum(-1))
    thr = max(4*SENSOR_NOISE, 0.30*np.percentile(d, 99))
    m = d > thr
    if k_open > 1:
        m = ndi.binary_opening(m, np.ones((k_open, k_open)))
        m = ndi.binary_closing(m, np.ones((k_open+1, k_open+1)))
    return m

def split_instances(mask, sigma=2.0, min_sep=14, peak_frac=0.55, min_area=120):
    '''Distance-transform watershed with non-maximum-suppressed seeds.'''
    d = ndi.distance_transform_edt(mask)
    ds = ndi.gaussian_filter(d, sigma)
    mx = ndi.maximum_filter(ds, size=2*min_sep+1)
    cand = mask & (ds >= mx - 1e-9)
    ys, xs = np.nonzero(cand)
    if len(ys) == 0:
        return ndi.label(mask)
    vals = ds[ys, xs]; o = np.argsort(-vals)
    ys, xs, vals = ys[o], xs[o], vals[o]
    dmax = ds[mask].max()
    keep = []
    for y, x, v in zip(ys, xs, vals):
        if v < peak_frac*dmax: continue
        if all((y-ky)**2 + (x-kx)**2 >= min_sep**2 for ky, kx in keep):
            keep.append((y, x))
    if len(keep) <= 1:
        return ndi.label(mask)
    markers = np.zeros(mask.shape, np.int32)
    for i, (y, x) in enumerate(keep, 1): markers[y, x] = i
    markers = ndi.grey_dilation(markers, size=3)
    lab = markers.copy(); lab[~mask] = 0
    pri = np.argsort(-ds[mask]); my, mx_ = np.nonzero(mask)
    my, mx_ = my[pri], mx_[pri]
    for _ in range(6):
        changed = 0
        for y, x in zip(my, mx_):
            if lab[y, x]: continue
            nb = lab[max(0, y-1):y+2, max(0, x-1):x+2]
            nb = nb[nb > 0]
            if nb.size:
                v, c = np.unique(nb, return_counts=True)
                lab[y, x] = v[np.argmax(c)]; changed += 1
        if not changed: break
    lab[~mask] = 0
    out = np.zeros_like(lab); k = 0
    for i in range(1, lab.max()+1):
        m = lab == i
        if m.sum() >= min_area: k += 1; out[m] = k
    return out, k

# ---- VET on geometry with known answers ------------------------------------
def _ellipse(shape, cx, cy, a, b, ang):
    yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
    X = (xx-cx)*np.cos(ang)+(yy-cy)*np.sin(ang)
    Y = -(xx-cx)*np.sin(ang)+(yy-cy)*np.cos(ang)
    return (X/a)**2 + (Y/b)**2 <= 1

_cases = [("2 well separated", [(120, 100, 55, 14, .2), (420, 220, 55, 14, -.2)]),
          ("3 fish, 2 touching", [(150, 140, 62, 15, .10), (252, 156, 62, 15, -.15),
                                  (400, 150, 55, 14, .30)]),
          ("4 in a chain", [(120, 120, 55, 14, .2), (215, 135, 55, 14, -.1),
                            (310, 150, 55, 14, .25), (405, 130, 55, 14, -.2)])]
rows = []
for nm, geoms in _cases:
    m = np.zeros((320, 560), bool)
    for g in geoms: m |= _ellipse(m.shape, *g)
    _, n_cc = ndi.label(m)
    _, n_ws = split_instances(m)
    rows.append(dict(case=nm, true=len(geoms), connected_components=n_cc, watershed=n_ws))
V = pd.DataFrame(rows).set_index("case")
display(V.style.set_caption("VET — instance splitting on geometry with known answers"))
assert (V.watershed == V.true).all(), "watershed is not recovering the true instance count"
print(f"[VET] watershed recovers the exact count in all {len(V)} cases, where connected")
print(f"      components under-counts by up to {int((V.true-V.connected_components).max())} "
      f"fish -> PASS")

FG = foreground(IMG)
LAB_CC, N_CC = ndi.label(FG)
LAB, N_WS = split_instances(FG, min_sep=13, peak_frac=0.42, min_area=110)
print(f"\nOn the rendered tank: {len(TRUTH)} true | {N_CC} connected components | {N_WS} watershed")

fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
ax[0].imshow(IMG); ax[0].set_title("input"); ax[0].axis("off")
ax[1].imshow(FG, cmap="gray"); ax[1].set_title("foreground"); ax[1].axis("off")
d = ndi.distance_transform_edt(FG)
ax[2].imshow(d, cmap="magma"); ax[2].set_title("distance transform (the ridges)"); ax[2].axis("off")
disp = np.zeros((*LAB.shape, 3))
for k in range(1, LAB.max()+1): disp[LAB == k] = RNG.random(3)*.8+.2
ax[3].imshow(disp); ax[3].set_title(f"{N_WS} instances"); ax[3].axis("off")
plt.tight_layout(); plt.show()

## Part 5 — Morphometrics: measure along the fish, not across it

### 5.1 The chord error

A fish bends. The PCA major-axis extent measures the **chord** of the bent body, and for a
circular-arc spine of bend angle $\theta$ the chord underestimates the arc length by exactly

$$\frac{\text{chord}}{\text{arc}} = \frac{\sin(\theta/2)}{\theta/2}$$

At 60° that is −4.5 %; at 100° it is −12 %. Cubed into mass: **−13 % and −30 %**. Bent fish
are systematically reported as lighter, and fish bend more when they are stressed or
crowded — so the bias is correlated with exactly the condition you are trying to detect.

### 5.2 Curvilinear length

Slice the mask along its principal axis, take the centroid of each slice, and sum the
polyline length through the centroids, extending at both ends to the extreme points. That
tracks the spine rather than the chord. The vetting below measures both against a tube of
**known arc length** across a range of bends.

In [ ]:
def _principal(m):
    ys, xs = np.nonzero(m)
    X = np.stack([xs-xs.mean(), ys-ys.mean()])
    ev, evec = np.linalg.eigh(np.cov(X))
    return X, evec[:, np.argmax(ev)], evec[:, np.argmin(ev)], (xs.mean(), ys.mean())

def chord_length(m):
    X, maj, _, _ = _principal(m); p = X.T @ maj
    return float(p.max()-p.min())

def curvilinear_length(m, n_slices=36):
    X, maj, mino, _ = _principal(m)
    p, q = X.T @ maj, X.T @ mino
    edges = np.linspace(p.min(), p.max(), n_slices+1)
    idx = np.clip(np.digitize(p, edges)-1, 0, n_slices-1)
    pts = [(p[idx == k].mean(), q[idx == k].mean()) for k in range(n_slices) if (idx == k).sum() >= 3]
    pts = np.array(pts)
    if len(pts) < 3: return chord_length(m)
    core = np.hypot(np.diff(pts[:, 0]), np.diff(pts[:, 1])).sum()
    return float(core + (p.max()-pts[-1, 0]) + (pts[0, 0]-p.min()))

def body_depth(m, n_slices=36):
    '''Maximum body depth (girth proxy) perpendicular to the spine.'''
    X, maj, mino, _ = _principal(m)
    p, q = X.T @ maj, X.T @ mino
    edges = np.linspace(p.min(), p.max(), n_slices+1)
    idx = np.clip(np.digitize(p, edges)-1, 0, n_slices-1)
    w = [q[idx == k].max()-q[idx == k].min() for k in range(n_slices) if (idx == k).sum() >= 3]
    return float(np.percentile(w, 90)) if w else 0.0

# ---- VET: tube of known arc length, swept over body bend -------------------
rows = []
S_true, r_px = 240.0, 13.0
truth_len = S_true + 2*r_px
for bend_deg in [0, 20, 40, 60, 80, 100]:
    bend = np.deg2rad(bend_deg)
    Hh, Ww = 420, 700
    yy, xx = np.mgrid[0:Hh, 0:Ww]; m = np.zeros((Hh, Ww), bool)
    if bend < 1e-6:
        t = np.linspace(-S_true/2, S_true/2, 900); sx, sy = 350+t, 210+0*t
    else:
        R = S_true/bend; t = np.linspace(-bend/2, bend/2, 900)
        sx, sy = 350+R*np.sin(t), 120+R*(1-np.cos(t))
    for X_, Y_ in zip(sx, sy): m |= ((xx-X_)**2 + (yy-Y_)**2) <= r_px*r_px
    Lc, Lv = chord_length(m), curvilinear_length(m)
    rows.append(dict(bend_deg=bend_deg, true_px=truth_len, chord_px=Lc, curvilinear_px=Lv,
                     chord_err_pct=100*(Lc/truth_len-1), curv_err_pct=100*(Lv/truth_len-1),
                     theory_chord_pct=100*(np.sin(bend/2)/(bend/2)-1) if bend > 1e-6 else 0.0,
                     chord_MASS_err_pct=100*((Lc/truth_len)**3-1)))
MORPH = pd.DataFrame(rows).set_index("bend_deg")
display(MORPH.round(2).style.background_gradient(subset=["chord_MASS_err_pct"], cmap="Reds_r")
        .set_caption("VET — chord vs curvilinear length on a tube of known arc length"))
assert MORPH.curv_err_pct.abs().max() < 2.5, "curvilinear estimator is biased"
assert MORPH.chord_err_pct.iloc[-1] < -8, "chord error is not reproducing the geometry"
print(f"[VET] curvilinear stays within {MORPH.curv_err_pct.abs().max():.1f}% across 0-100 deg of")
print(f"      bend, while the chord drifts to {MORPH.chord_err_pct.iloc[-1]:.1f}% "
      f"({MORPH.chord_MASS_err_pct.iloc[-1]:.0f}% in mass)")
print("      and the chord error tracks the closed form sin(t/2)/(t/2) -> PASS")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(MORPH.index, MORPH.chord_err_pct, "o-", color="#C0392B", label="chord / PCA axis")
ax[0].plot(MORPH.index, MORPH.theory_chord_pct, "--", color="k", lw=.9,
           label="theory sin(t/2)/(t/2)")
ax[0].plot(MORPH.index, MORPH.curv_err_pct, "s-", color="#1F3A5F", label="curvilinear")
ax[0].axhline(0, color="k", lw=.6)
ax[0].set_xlabel("body bend (deg)"); ax[0].set_ylabel("length error (%)")
ax[0].legend(fontsize=7); ax[0].set_title("Length estimator bias vs body bend")
ax[1].plot(MORPH.index, MORPH.chord_MASS_err_pct, "o-", color="#C0392B")
ax[1].axhline(0, color="k", lw=.6)
ax[1].set_xlabel("body bend (deg)"); ax[1].set_ylabel("resulting MASS error (%)")
ax[1].set_title("Cubed into biomass")
plt.tight_layout(); plt.show()

In [ ]:
def measure_instances(lab, img, f_px=CAM["f_px"], Z_known=None, n_water=N_WATER):
    '''Convert each instance mask into physical measurements.
    Z_known: per-instance range (from stereo in a real rig). If None, a single nominal
    range is used and the resulting bias is reported honestly by the caller.'''
    out = []
    for k in range(1, lab.max()+1):
        m = lab == k
        if m.sum() < 80: continue
        ys, xs = np.nonzero(m)
        Lpx_chord = chord_length(m); Lpx_curv = curvilinear_length(m); dpx = body_depth(m)
        Z = float(Z_known[k-1]) if Z_known is not None else 150.0
        scale = px_per_cm(f_px, Z, n_water)
        L_cm = Lpx_curv/scale
        out.append(dict(inst=k, L_cm=L_cm, L_cm_chord=Lpx_chord/scale,
                        depth_cm=dpx/scale, aspect=Lpx_curv/max(dpx, 1e-6),
                        Z_cm=Z, area_px=int(m.sum()),
                        cx=float(xs.mean()), cy=float(ys.mean()), mask=m))
    return out

# give the pipeline the true range per instance (a stereo rig supplies this)
def match_to_truth(insts, truth):
    if not insts or not truth: return []
    C = np.zeros((len(insts), len(truth)))
    for i, e in enumerate(insts):
        for j, t in enumerate(truth):
            inter = (e["mask"] & t["mask"]).sum(); union = (e["mask"] | t["mask"]).sum()
            C[i, j] = -inter/max(union, 1)
    r, c = linear_sum_assignment(C)
    return [(int(a), int(b), float(-C[a, b])) for a, b in zip(r, c) if -C[a, b] > 0.25]

pairs = match_to_truth(measure_instances(LAB, IMG), TRUTH)
Zs = np.full(LAB.max(), 150.0)
for a, b, _ in pairs: Zs[a] = TRUTH[b]["Z_cm"]
INST = measure_instances(LAB, IMG, Z_known=Zs)
pairs = match_to_truth(INST, TRUTH)
print(f"{len(pairs)} of {len(TRUTH)} true fish matched at IoU > 0.25")

rows = []
for a, b, iou in pairs:
    e, t = INST[a], TRUTH[b]
    rows.append(dict(iou=iou, L_true=t["L_cm"], L_est=e["L_cm"], L_chord=e["L_cm_chord"],
                     bend_deg=t["bend_deg"], occlusion=t["occlusion"],
                     mass_true=t["mass_g"], mass_est=float(mass_from_length(e["L_cm"]))))
M = pd.DataFrame(rows)
M["L_err_pct"] = 100*(M.L_est/M.L_true-1)
M["L_chord_err_pct"] = 100*(M.L_chord/M.L_true-1)
M["mass_err_pct"] = 100*(M.mass_est/M.mass_true-1)
display(M.round(2).head(10))
print(f"\nlength : bias {M.L_err_pct.mean():+.2f}%  MAPE {M.L_err_pct.abs().mean():.2f}%  "
      f"(chord would be bias {M.L_chord_err_pct.mean():+.2f}%)")
print(f"mass   : bias {M.mass_err_pct.mean():+.2f}%  MAPE {M.mass_err_pct.abs().mean():.2f}%")
print("\nThe mass MAPE is much larger than the length MAPE and always will be: it carries the")
print("b=3 amplification PLUS the biological scatter around the length-weight relationship,")
print("which no camera can see. Individual mass is intrinsically noisy; POPULATION biomass is")
print("the estimable quantity, and Part 6 is about estimating it without bias.")

## Part 6 — Biomass: three corrections that are usually missed

### 6.1 Jensen / smearing correction

The length-weight relationship is fitted in log space, so residuals are log-normal with
scatter $\sigma$. Then $\mathbb{E}[W] \ne a\,\mathbb{E}[L]^{b}$; the correct estimator carries
the **smearing factor**

$$\mathbb{E}[W\mid L] = a L^{b}\,e^{\sigma^{2}/2}$$

### 6.2 Measurement-error inflation

If measured length is $L' = L e^{\varepsilon}$ with $\varepsilon\sim N(0,s^2)$, then
$\mathbb{E}[L'^{b}] = \mathbb{E}[L^{b}]\,e^{b^{2}s^{2}/2}$, so **random length noise inflates
mean mass** — upward, not symmetrically. At $s=0.10$ and $b=3$ that is +4.7 %. We verify the
formula against simulation.

### 6.3 Size-selectivity correction

Small and distant fish fall below the detection floor, so the measured sample is
**size-biased**. If detection probability is $\pi(L)$, the unbiased population mean uses
Horvitz–Thompson weights $1/\pi(L)$:

$$\widehat{\bar W} = \frac{\sum_i W_i/\pi(L_i)}{\sum_i 1/\pi(L_i)},\qquad
 \widehat{N} = \sum_i \frac{1}{\pi(L_i)}$$

$\pi(L)$ is estimable from the renderer here, and in a real tank from a subsample measured by
a second method (a grading run, or a stereo rig with a wider field). Skipping it inflates mean
mass and **under**-counts the stock — the two errors that together make a tank look safely
stocked when it is not.

In [ ]:
# ---- VET the two log-normal corrections against simulation -----------------
r = np.random.default_rng(1)
Lp = r.normal(28, 4, 300000); Lp = Lp[Lp > 8]
rows = []
for s in [0.05, 0.10, 0.15, 0.20]:
    Lm = Lp*np.exp(r.normal(0, s, Lp.size))
    emp = np.mean(mass_from_length(Lm))/np.mean(mass_from_length(Lp)) - 1
    thy = np.exp(LW["b"]**2*s**2/2) - 1
    rows.append(dict(ln_length_noise_sd=s, empirical_mass_inflation_pct=100*emp,
                     theory_exp_b2s2_2_pct=100*thy,
                     smearing_factor_exp_s2_2=float(np.exp(s**2/2))))
JB = pd.DataFrame(rows).set_index("ln_length_noise_sd")
display(JB.round(4).style.set_caption("VET — Jensen corrections, theory vs simulation"))
assert (JB.empirical_mass_inflation_pct - JB.theory_exp_b2s2_2_pct).abs().max() < 0.6
print("[VET] measurement-noise inflation matches exp(b^2 s^2 / 2) to better than 0.6 pp -> PASS")

# ---- size selectivity, estimated from the renderer -------------------------
def detection_curve(n_frames=40, seed=0):
    '''Render frames, segment them, and record which true fish were recovered, by length.'''
    rr = np.random.default_rng(seed)
    seen, allf = [], []
    for i in range(n_frames):
        im, tr = render_tank(n_fish=int(rr.integers(10, 20)), seed=int(rr.integers(1e9)))
        lab, _ = split_instances(foreground(im), min_sep=13, peak_frac=0.42, min_area=110)
        ins = measure_instances(lab, im)
        pr = match_to_truth(ins, tr)
        got = {b for _, b, _ in pr}
        for j, t in enumerate(tr):
            allf.append(t["L_cm"]); seen.append(j in got)
    return np.array(allf), np.array(seen)

NF = 25 if RUN["FAST"] else 70
L_all, seen = detection_curve(NF, seed=RUN["SEED"])
bins = np.linspace(np.percentile(L_all, 1), np.percentile(L_all, 99), 9)
bi = np.clip(np.digitize(L_all, bins)-1, 0, len(bins)-2)
pi_L = np.array([seen[bi == k].mean() if (bi == k).sum() >= 5 else np.nan
                 for k in range(len(bins)-1)])
ctr = 0.5*(bins[1:]+bins[:-1])
ok = ~np.isnan(pi_L)
sl, ic = np.polyfit(ctr[ok], np.log(np.clip(pi_L[ok], .02, .999)/(1-np.clip(pi_L[ok], .02, .999))), 1)
def pi_of_L(L): return 1/(1+np.exp(-(ic+sl*np.asarray(L, float))))

print(f"\ndetection probability from {NF} rendered frames, {len(L_all)} true fish:")
print(pd.DataFrame(dict(L_cm=ctr.round(1), pi_hat=np.round(pi_L, 3))).to_string(index=False))
print(f"logistic fit: pi(L) = sigmoid({ic:.2f} + {sl:.3f} L)")

# ---- assemble a population estimate with all three corrections -------------
def biomass_estimate(L_cm, sigma_lnW=LW["sigma_lnW"], use_pi=True, use_smear=True,
                     use_noise=False, s_meas=0.0, b=LW["b"]):
    L_cm = np.asarray(L_cm, float)
    W = mass_from_length(L_cm)
    if use_smear: W = W*np.exp(sigma_lnW**2/2)
    if use_noise: W = W/np.exp(b**2*s_meas**2/2)       # remove the inflation, not add it
    w = 1/np.clip(pi_of_L(L_cm), 0.05, 1.0) if use_pi else np.ones_like(L_cm)
    return dict(N_hat=float(w.sum()), mean_mass_g=float((W*w).sum()/w.sum()),
                total_biomass_g=float((W*w).sum()))

# ground truth for the WHOLE rendered population across the same frames
rr = np.random.default_rng(RUN["SEED"])
pop_true_L, pop_meas_L = [], []
for i in range(NF):
    im, tr = render_tank(n_fish=int(rr.integers(10, 20)), seed=int(rr.integers(1e9)))
    lab, _ = split_instances(foreground(im), min_sep=13, peak_frac=0.42, min_area=110)
    ins = measure_instances(lab, im)
    pr = match_to_truth(ins, tr)
    Zs = np.full(max(lab.max(), 1), 150.0)
    for a, b_, _ in pr: Zs[a] = tr[b_]["Z_cm"]
    ins = measure_instances(lab, im, Z_known=Zs)
    pr = match_to_truth(ins, tr)
    pop_true_L += [t["L_cm"] for t in tr]
    pop_meas_L += [ins[a]["L_cm"] for a, _, _ in pr]
pop_true_L = np.array(pop_true_L); pop_meas_L = np.array(pop_meas_L)
truth_total = float(np.sum(mass_from_length(pop_true_L)*np.exp(LW["sigma_lnW"]**2/2)))
truth_N = len(pop_true_L)

rows = []
for nm, kw in [("raw (no corrections)", dict(use_pi=False, use_smear=False)),
               ("+ smearing only", dict(use_pi=False, use_smear=True)),
               ("+ size-selectivity only", dict(use_pi=True, use_smear=False)),
               ("all corrections", dict(use_pi=True, use_smear=True))]:
    e = biomass_estimate(pop_meas_L, **kw)
    rows.append(dict(estimator=nm, N_hat=e["N_hat"], N_true=truth_N,
                     count_err_pct=100*(e["N_hat"]/truth_N-1),
                     mean_mass_g=e["mean_mass_g"],
                     total_kg=e["total_biomass_g"]/1000, true_kg=truth_total/1000,
                     biomass_err_pct=100*(e["total_biomass_g"]/truth_total-1)))
BIO = pd.DataFrame(rows).set_index("estimator")
display(BIO.round(2).style.background_gradient(subset=["biomass_err_pct"], cmap="RdYlGn_r")
        .set_caption("Biomass estimate with each correction switched on"))

# ---- cluster bootstrap CI on total biomass ---------------------------------
def boot_biomass(L, n_boot=600, seed=0):
    r = np.random.default_rng(seed); out = []
    for _ in range(n_boot):
        s = r.choice(L, len(L), replace=True)
        out.append(biomass_estimate(s)["total_biomass_g"]/1000)
    return np.percentile(out, [2.5, 50, 97.5])
lo, mid, hi = boot_biomass(pop_meas_L, 300 if RUN["FAST"] else 1500, RUN["SEED"])
print(f"\ntotal biomass: {mid:.2f} kg, 95% CI [{lo:.2f}, {hi:.2f}] kg "
      f"(+/-{100*(hi-lo)/2/mid:.1f}%) | true {truth_total/1000:.2f} kg")
print("Report the interval. A stocking-density decision taken on a point estimate that is")
print("actually +/-8% is a decision taken without knowing whether you are over the limit.")

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.2))
ax[0].plot(ctr, pi_L, "o", color="#1F3A5F"); 
LL = np.linspace(ctr.min()-3, ctr.max()+3, 120)
ax[0].plot(LL, pi_of_L(LL), color="#C0392B", label="logistic fit")
ax[0].set_xlabel("true length (cm)"); ax[0].set_ylabel("detection probability")
ax[0].legend(fontsize=7); ax[0].set_title("Small fish are missed")
ax[1].hist(pop_true_L, bins=24, alpha=.55, label="true population", color="#1F3A5F")
ax[1].hist(pop_meas_L, bins=24, alpha=.55, label="measured sample", color="#C0392B")
ax[1].set_xlabel("length (cm)"); ax[1].legend(fontsize=7)
ax[1].set_title("The sample is size-biased")
ax[2].barh(BIO.index, BIO.biomass_err_pct, color=["#C0392B", "#E08A5A", "#7FA6C9", "#1F3A5F"])
ax[2].axvline(0, color="k", lw=.8)
ax[2].set_xlabel("biomass error (%)"); ax[2].tick_params(labelsize=6.5)
ax[2].set_title("Each correction, measured")
plt.tight_layout(); plt.show()

## Part 7 — Welfare: a weighted index, not a threshold

### 7.1 Why a single condition factor is not a welfare assessment

Fulton's condition factor

$$K = \frac{100\,W}{L^{3}}$$

is one indicator among many, and it is **scale-dependent** — $K$ drifts with size even in
healthy fish, because real length-weight exponents are not exactly 3. Published operational
frameworks (SWIM 1.0, Stien et al. 2013; SWIM 2.0; FISHWELL; LAKSVEL) instead combine
**Operational Welfare Indicators** across three levels — environment, group, individual —
into a weighted score.

$$\mathrm{OWI}_{total} = \sum_{i} \mathrm{RWF}_i \cdot \mathrm{IS}_i,
\qquad \mathrm{RWF}_i = \frac{\mathrm{WF}_i}{\sum_j \mathrm{WF}_j}$$

with $\mathrm{IS}_i$ the indicator score (0 = good, 1 = severe) and $\mathrm{WF}_i$ the
indicator's weight.

### 7.2 What a camera can and cannot score

| SWIM indicator | Camera-derivable? | How, here |
|---|---|---|
| condition factor | **yes** (via L and W estimate) | Part 6, with its CI |
| emaciation / body shape | **yes** | length-to-depth aspect ratio |
| vertical/spatial distribution | **yes** | instance centroids |
| swimming speed and polarisation | **yes** (needs tracking) | Part 8 |
| fin and skin condition | partly — needs resolution we do not have at range | flagged, not scored |
| gill condition, maturation | **no** | requires handling; out of camera scope |
| temperature, DO, salinity | **no** — from sensors | supplied by the thermal handoff |
| mortality rate, appetite | **no** — from husbandry records | operator input |

**This is the honest statement the report needs**: vision covers roughly half the SWIM
indicator set. An index computed from vision alone is a *partial* welfare score and must be
labelled as such, with the missing indicators listed rather than silently weighted to zero.

In [ ]:
# Indicator weights: PLACEHOLDERS. Set with veterinary and aquatic-science input.
OWI = pd.DataFrame([
 ("condition_factor",      "individual",  "vision", 8, "Fulton K outside the species band"),
 ("body_shape_aspect",     "individual",  "vision", 6, "length:depth ratio outside normal"),
 ("size_heterogeneity",    "group",       "vision", 5, "robust CV of length across the stock"),
 ("spatial_crowding",      "group",       "vision", 5, "clustering vs uniform occupancy"),
 ("swimming_activity",     "group",       "vision", 6, "mean speed outside the normal band"),
 ("schooling_polarisation","group",       "vision", 4, "loss of alignment = stress or hypoxia"),
 ("temperature",           "environment", "sensor", 9, "vs species thermal optimum"),
 ("dissolved_oxygen",      "environment", "sensor", 10, "vs species hypoxia threshold"),
 ("unionised_ammonia",     "environment", "sensor", 8, "NH3 from TAN, pH and temperature"),
 ("fin_and_skin",          "individual",  "NOT COVERED", 7, "needs close-range imaging"),
 ("gill_condition",        "individual",  "NOT COVERED", 8, "needs handling"),
 ("mortality_rate",        "group",       "NOT COVERED", 10, "husbandry record"),
 ("appetite",              "group",       "NOT COVERED", 7, "feeding-system record"),
], columns=["indicator", "level", "source", "weight", "note"])
OWI["RWF"] = (OWI.weight/OWI.weight.sum()).round(4)
display(OWI.style.apply(lambda s: ["background-color:#ffe0e0" if v == "NOT COVERED"
                                   else "" for v in s] if s.name == "source" else ["" for _ in s]))
cov = OWI[OWI.source != "NOT COVERED"].weight.sum()/OWI.weight.sum()
vis_only = OWI[OWI.source == "vision"].weight.sum()/OWI.weight.sum()
print(f"weight coverage with vision + sensors: {100*cov:.0f}% of the SWIM-style weight")
print(f"weight coverage with VISION ALONE    : {100*vis_only:.0f}%")
print("The uncovered 32% is not small and is not random: it is mortality, appetite, gills and")
print("fins — exactly the indicators a vet would look at first. Say so in the report.")

K_BAND = (1.05, 1.55)          # placeholder, species/life-stage specific
ASPECT_BAND = (3.2, 5.2)
SPEED_BAND = (0.6, 2.2)        # body lengths per second

def score_linear(v, lo, hi, hard_lo=None, hard_hi=None):
    '''0 inside the good band, rising to 1 at the hard limits.'''
    hard_lo = lo - (hi-lo) if hard_lo is None else hard_lo
    hard_hi = hi + (hi-lo) if hard_hi is None else hard_hi
    if lo <= v <= hi: return 0.0
    if v < lo:  return float(np.clip((lo-v)/max(lo-hard_lo, 1e-9), 0, 1))
    return float(np.clip((v-hi)/max(hard_hi-hi, 1e-9), 0, 1))

def welfare_index(individuals, group, env, owi=OWI):
    '''individuals: DataFrame with K and aspect. group/env: dicts. Returns score + breakdown.'''
    w = owi.set_index("indicator")
    covered = owi[owi.source != "NOT COVERED"].indicator.tolist()
    rwf = (w.loc[covered, "weight"]/w.loc[covered, "weight"].sum())
    s = {}
    s["condition_factor"] = float(np.mean([score_linear(k, *K_BAND) for k in individuals.K]))
    s["body_shape_aspect"] = float(np.mean([score_linear(a, *ASPECT_BAND) for a in individuals.aspect]))
    s["size_heterogeneity"] = score_linear(group["cv_length"], 0.0, 0.18, 0.0, 0.40)
    s["spatial_crowding"] = score_linear(group["clustering"], 0.0, 1.25, 0.0, 2.5)
    s["swimming_activity"] = score_linear(group["speed_BL_s"], *SPEED_BAND)
    s["schooling_polarisation"] = score_linear(group["polarisation"], 0.35, 0.95, 0.0, 1.0)
    s["temperature"] = score_linear(env["T_C"], env["T_opt_lo"], env["T_opt_hi"],
                                    env["T_opt_lo"]-6, env["T_CTM"])
    s["dissolved_oxygen"] = score_linear(env["DO_mgL"], env["DO_good"], 20.0,
                                         env["DO_crit"], 25.0)
    s["unionised_ammonia"] = score_linear(env["NH3_mgL"], 0.0, 0.012, 0.0, 0.05)
    total = float(sum(rwf[k]*s[k] for k in covered))
    br = pd.DataFrame(dict(indicator=covered, score=[s[k] for k in covered],
                           RWF=[float(rwf[k]) for k in covered]))
    br["contribution"] = (br.score*br.RWF).round(4)
    return total, br.sort_values("contribution", ascending=False)

def grade(x): return "good" if x < .15 else "acceptable" if x < .30 else \
                     "marginal — review" if x < .50 else "poor — escalate now"
print(f"\nSeverity bands (placeholders): {[(b, grade(b)) for b in [0.05,0.2,0.35,0.6]]}")

In [ ]:
# individual-level scores from the measured instances
IND = pd.DataFrame([dict(L_cm=e["L_cm"], depth_cm=e["depth_cm"], aspect=e["aspect"])
                    for e in INST if e["depth_cm"] > 0])
IND["mass_g"] = mass_from_length(IND.L_cm)*np.exp(LW["sigma_lnW"]**2/2)
IND["K"] = fulton_K(IND.mass_g, IND.L_cm)

def robust_cv(x):
    x = np.asarray(x, float); med = np.median(x)
    return float(1.4826*np.median(np.abs(x-med))/max(med, 1e-9))

def clustering_index(cx, cy, W, H):
    '''Clark-Evans style: mean nearest-neighbour distance vs the random expectation.
    <1 clustered, ~1 random, >1 regular. We report the INVERSE so higher = more crowded.'''
    P = np.stack([cx, cy], 1); n = len(P)
    if n < 3: return 1.0
    D = np.linalg.norm(P[:, None] - P[None], axis=-1) + np.eye(n)*1e9
    r_obs = D.min(1).mean()
    r_exp = 0.5/np.sqrt(n/(W*H))
    return float(r_exp/max(r_obs, 1e-9))

GROUP = dict(cv_length=robust_cv(IND.L_cm),
             clustering=clustering_index(np.array([e["cx"] for e in INST]),
                                         np.array([e["cy"] for e in INST]), W_IMG, H_IMG),
             speed_BL_s=1.1, polarisation=0.72)     # replaced by tracking in Part 8

Tloop = AQ["loop_inlet_T_C"]
ENV = dict(T_C=Tloop, T_opt_lo=12.0, T_opt_hi=16.0, T_CTM=25.1,
           DO_mgL=AQ["DO_at_loop_T_mgL"], DO_good=7.0, DO_crit=5.0,
           NH3_mgL=0.006)
score, BR = welfare_index(IND, GROUP, ENV)
print(f"group state: CV(length)={GROUP['cv_length']:.3f}, crowding={GROUP['clustering']:.2f}")
print(f"environment from the thermal handoff: T={ENV['T_C']:.1f} C, DO={ENV['DO_mgL']:.2f} mg/L")
print(f"\nPARTIAL welfare index = {score:.3f}  ->  {grade(score)}")
print(f"(covers {100*cov:.0f}% of the SWIM-style indicator weight; see the NOT COVERED rows)")
display(BR.round(4).style.background_gradient(subset=["contribution"], cmap="Reds"))

# sensitivity: how does the index move with the loop temperature the thermal notebook sets?
TT = np.linspace(10, 24, 60)
sc = []
for t in TT:
    e = dict(ENV); e["T_C"] = t
    # DO falls with warming; use the thermal notebook's own relationship shape
    e["DO_mgL"] = AQ["DO_at_loop_T_mgL"]*(1 - 0.021*(t-Tloop))
    e["NH3_mgL"] = 0.006*2.0**((t-Tloop)/12.0)          # NH3 fraction roughly doubles per 12 C
    sc.append(welfare_index(IND, GROUP, e)[0])
sc = np.array(sc)

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.2))
ax[0].hist(IND.K, bins=12, color="#1F3A5F", edgecolor="w")
for v, c in [(K_BAND[0], "crimson"), (K_BAND[1], "crimson")]:
    ax[0].axvline(v, color=c, ls="--", lw=1)
ax[0].set_xlabel("Fulton's K"); ax[0].set_title("Condition factor vs the acceptable band")
ax[1].barh(BR.indicator, BR.contribution, color="#C0392B")
ax[1].set_xlabel("contribution to the index"); ax[1].tick_params(labelsize=6.5)
ax[1].set_title("What is driving the score")
ax[2].plot(TT, sc, color="#1F3A5F", lw=1.8)
for y, c, l in [(.15, "#2E7D82", "good"), (.30, "#8E6C1F", "acceptable"), (.50, "#C0392B", "marginal")]:
    ax[2].axhline(y, ls=":", color=c, lw=1)
ax[2].axvline(Tloop, color="k", ls="--", lw=1)
ax[2].annotate(f"loop inlet\n{Tloop:.1f} C", (Tloop+.3, sc.max()*.75), fontsize=7)
ax[2].set_xlabel("loop inlet temperature (C)"); ax[2].set_ylabel("partial welfare index")
ax[2].set_title("Welfare vs the temperature the reuse loop delivers")
plt.tight_layout(); plt.show()
T_lim = TT[np.argmax(sc > 0.30)] if (sc > 0.30).any() else np.nan
print(f"\nThe index crosses 'marginal' at about {T_lim:.1f} C. THAT is the number the thermal")
print("notebook needs back: it is the ceiling on the loop inlet temperature, and therefore a")
print("constraint on how much captured heat the loop can absorb. The two notebooks close a loop.")
print("It is a PLACEHOLDER ceiling until the bands are set with veterinary input.")

## Part 8 — Tracking: individual growth, de-duplicated counts, behaviour

Three things tracking buys that single frames cannot give:

1. **Counting without double-counting.** A fish seen in 40 frames is one fish. Track-level
   counts feed Part 6's $\hat N$; frame-level counts do not.
2. **Behaviour indicators.** Swimming speed (body lengths per second) and schooling
   **polarisation**
   $$\Phi = \left\|\frac{1}{n}\sum_{i=1}^{n}\hat{v}_i\right\| \in [0,1]$$
   are established group welfare indicators. $\Phi \to 1$ is a tight polarised school;
   collapse toward 0 with rising speed is a recognised stress signature.
3. **Repeated measurement of the same individual**, which averages down the per-fish
   measurement noise from Part 2 by $\sqrt{n_{\text{obs}}}$.

In [ ]:
class KalmanTrack:
    def __init__(self, z0, dt=1., q=9., r=4.):
        self.x = np.array([z0[0], z0[1], 0., 0.])
        self.P = np.diag([r, r, 400., 400.])
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]], float)
        G = np.array([[dt*dt/2,0],[0,dt*dt/2],[dt,0],[0,dt]])
        self.Q = G @ (q*np.eye(2)) @ G.T
        self.H = np.array([[1,0,0,0],[0,1,0,0]], float); self.R = r*np.eye(2)
        self.miss = 0; self.L = []; self.zs = []
    def predict(self): self.x = self.F@self.x; self.P = self.F@self.P@self.F.T+self.Q; return self.x[:2]
    def gate(self, z):
        S = self.H@self.P@self.H.T+self.R; y = np.asarray(z)-self.H@self.x
        return float(y@np.linalg.inv(S)@y)
    def update(self, z, L=None):
        S = self.H@self.P@self.H.T+self.R; K = self.P@self.H.T@np.linalg.inv(S)
        self.x = self.x + K@(np.asarray(z)-self.H@self.x)
        self.P = (np.eye(4)-K@self.H)@self.P; self.miss = 0
        self.zs.append(np.asarray(z, float))
        if L is not None: self.L.append(L)

CHI2 = 9.21

ARENA_W, ARENA_H = 900, 600          # px; larger than one frame, see the comment below

def simulate_swim(n_fish=12, T=60, speed_BL=1.1, polarisation=0.75, L_cm=27.0,
                  Z=400., fps=25.0, seed=0, p_det=0.92, jitter=0.06):
    '''Ground-truth trajectories with a controlled polarisation, then noisy detections.'''
    r = np.random.default_rng(seed)
    sc = px_per_cm(CAM["f_px"], Z)
    v_mag = speed_BL*L_cm*sc/fps                        # px per frame
    # Z=400 cm gives a ~98 cm field of view: wide enough that a fish crossing it takes
    # tens of frames. At Z=150 cm the view is only 37 cm and a 1 BL/s fish moves 34 px
    # per frame, which no constant-velocity tracker can follow -- a rig geometry problem,
    # not a tracker problem, and worth stating as a camera-siting requirement.
    school_dir = r.uniform(0, 2*np.pi)
    dirs = school_dir + r.normal(0, np.sqrt(max(1e-6, -2*np.log(max(polarisation, 1e-6)))), n_fish)
    # The tracking arena is larger than one camera frame and fish start centrally, so the
    # reflecting boundary is rarely reached. Wall bounces inject heading reversals that have
    # nothing to do with fish behaviour and would corrupt the polarisation indicator.
    P = np.stack([r.uniform(.2*ARENA_W, .8*ARENA_W, n_fish),
                  r.uniform(.2*ARENA_H, .8*ARENA_H, n_fish)], 1)
    n_bounce = 0
    V = np.stack([np.cos(dirs), np.sin(dirs)], 1)*v_mag
    dets, gt = [], []
    for t in range(T):
        P = P + V + r.normal(0, jitter*v_mag, P.shape)
        for i in range(n_fish):
            for d in (0, 1):
                lim = (ARENA_W, ARENA_H)[d]
                if P[i, d] < 8 or P[i, d] > lim-8:
                    V[i, d] *= -1; P[i, d] = np.clip(P[i, d], 8, lim-8); n_bounce += 1
        gt.append(P.copy())
        Z_t = []
        for i in range(n_fish):
            if r.random() < p_det:
                Z_t.append([P[i, 0]+r.normal(0, 1.5), P[i, 1]+r.normal(0, 1.5), i,
                            L_cm*np.exp(r.normal(0, .05))])
        dets.append(np.array(Z_t) if Z_t else np.zeros((0, 4)))
    return dets, np.array(gt), dict(true_polarisation=polarisation, true_speed_BL=speed_BL,
                                    n_bounce=n_bounce)

def run_tracker(dets, max_miss=5):
    tracks, nid, assoc = {}, 0, []
    for t, Zt in enumerate(dets):
        for tr in tracks.values(): tr.predict()
        ids = list(tracks); used = set()
        if ids and len(Zt):
            C = np.full((len(ids), len(Zt)), 1e6)
            for a, k in enumerate(ids):
                for b, z in enumerate(Zt):
                    d = tracks[k].gate(z[:2])
                    if d < CHI2: C[a, b] = d
            ra, cb = linear_sum_assignment(C)
            for a, b in zip(ra, cb):
                if C[a, b] < CHI2:
                    tracks[ids[a]].update(Zt[b][:2], Zt[b][3]); used.add(b)
                    assoc.append((t, ids[a], int(Zt[b][2])))
        for b in range(len(Zt)):
            if b not in used:
                tracks[nid] = KalmanTrack(Zt[b][:2]); tracks[nid].L.append(Zt[b][3])
                assoc.append((t, nid, int(Zt[b][2]))); nid += 1
        for k in list(tracks):
            if k not in [a[1] for a in assoc if a[0] == t]:
                tracks[k].miss += 1
                if tracks[k].miss > max_miss: del tracks[k]
    return np.array(assoc), nid, tracks

def polarisation_from_velocity(V):
    '''Mean resultant length of the heading unit vectors, in [0,1].'''
    U = V/np.maximum(np.linalg.norm(V, axis=-1, keepdims=True), 1e-9)
    return float(np.linalg.norm(U.mean(axis=0)))

def polarisation_from_tracks(tracks, min_hits=6):
    '''Use the KALMAN velocity estimates, not raw frame-to-frame differences.
    Raw differencing is dominated by localisation noise and systematically UNDER-reports
    polarisation; the filter is what makes the indicator usable.'''
    V = np.array([t.x[2:] for t in tracks.values() if len(t.L) >= min_hits])
    return polarisation_from_velocity(V) if len(V) >= 3 else np.nan

def polarisation_raw(tracks, min_hits=6):
    '''Unfiltered baseline: difference the last two DETECTIONS of each track. This is the
    fair comparison -- differencing ground-truth positions instead would flatter it with
    information no real system has.'''
    V = np.array([t.zs[-1]-t.zs[-2] for t in tracks.values()
                  if len(t.L) >= min_hits and len(t.zs) >= 2])
    return polarisation_from_velocity(V) if len(V) >= 3 else np.nan

# Several seeds per state: a behaviour indicator asserted on ONE random draw is a
# coin flip, not a measurement. Replicates also give the spread a reviewer needs.
N_REP = 4 if RUN["FAST"] else 10
rows = []
for pol, spd, nm in [(0.90, 1.0, "calm, tight school"), (0.60, 1.4, "moderate"),
                     (0.25, 2.6, "erratic / stressed")]:
    rec = []
    for rep in range(N_REP):
        dets, gt, meta = simulate_swim(polarisation=pol, speed_BL=spd,
                                       seed=RUN["SEED"]+100*rep)
        A, ntr, tracks = run_tracker(dets)
        real = A[A[:, 2] >= 0]
        pur = []
        for g in np.unique(real[:, 2]):
            _, c = np.unique(real[real[:, 2] == g, 1], return_counts=True)
            pur.append(c.max()/c.sum())
        Lhat = np.array([np.mean(t.L) for t in tracks.values() if len(t.L) >= 5])
        rec.append(dict(phi_k=polarisation_from_tracks(tracks),
                        phi_r=polarisation_raw(tracks),
                        ntr=ntr, purity=float(np.mean(pur)),
                        n5=len(Lhat), sd_avg=float(np.std(Lhat)) if len(Lhat) > 1 else np.nan,
                        bounce=meta["n_bounce"]))
    R = pd.DataFrame(rec)
    rows.append(dict(state=nm, true_polarisation=pol, speed_BL_s=spd,
                     polarisation_kalman=round(float(R.phi_k.mean()), 3),
                     polarisation_kalman_sd=round(float(R.phi_k.std()), 3),
                     polarisation_raw_diff=round(float(R.phi_r.mean()), 3),
                     true_fish=12, tracks_created=round(float(R.ntr.mean()), 1),
                     count_err_pct=round(100*(R.ntr.mean()-12)/12, 1),
                     wall_bounces=round(float(R.bounce.mean()), 1),
                     mean_track_purity=round(float(R.purity.mean()), 3),
                     n_with_5plus_obs=round(float(R.n5.mean()), 1),
                     length_sd_single=round(27*0.05, 3),
                     length_sd_averaged=round(float(R.sd_avg.mean()), 3)))
TRK = pd.DataFrame(rows).set_index("state")
display(TRK.style.background_gradient(subset=["mean_track_purity"], cmap="RdYlGn")
        .set_caption("Tracking: counts, behaviour and measurement averaging"))
# The tracker is REQUIRED to hold together on calm and moderate schools. Degradation on the
# erratic case is the expected, documented failure -- asserting it away would hide the very
# effect this cell exists to show.
_ok = TRK.loc[["calm, tight school", "moderate"]]
assert _ok.mean_track_purity.min() > 0.80, "tracker fragments even on a well-behaved school"
# A behaviour indicator read through a FRAGMENTING tracker is not a measurement. Gate it.
PURITY_GATE = 0.80
TRK["behaviour_reportable"] = TRK.mean_track_purity >= PURITY_GATE
_rep = TRK[TRK.behaviour_reportable]
assert _rep.polarisation_kalman.is_monotonic_decreasing, \
    "polarisation does not follow the injected ordering where tracking is trustworthy"
print(f"[VET] where track purity >= {PURITY_GATE}, polarisation from Kalman velocities follows")
print(f"      the injected ordering {_rep.true_polarisation.tolist()} -> "
      f"{_rep.polarisation_kalman.tolist()} -> PASS")

# I expected the Kalman velocity to beat single-frame differencing. It does not, here.
_k, _r = _rep.polarisation_kalman.mean(), _rep.polarisation_raw_diff.mean()
print(f"\nKalman {_k:.3f} vs raw single-frame differencing {_r:.3f}: raw is "
      f"{'higher' if _r > _k else 'lower'} under THESE conditions.")
print("That is not what I assumed when writing the filter in, so here is the actual")
print("trade-off, measured rather than asserted:")
_sw = []
for _noise in [0.5, 1.5, 3.0, 6.0, 10.0]:
    _kk, _rr = [], []
    for _rep_i in range(3):
        _d, _g, _m = simulate_swim(polarisation=0.80, speed_BL=1.0,
                                   seed=RUN["SEED"]+7*_rep_i)
        _d = [np.array([[z[0]+np.random.default_rng(int(z[2])+_rep_i*97).normal(0, _noise),
                         z[1]+np.random.default_rng(int(z[2])+_rep_i*31).normal(0, _noise),
                         z[2], z[3]] for z in Z_]) if len(Z_) else Z_ for Z_ in _d]
        _A, _n, _t = run_tracker(_d)
        _kk.append(polarisation_from_tracks(_t)); _rr.append(polarisation_raw(_t))
    _sw.append(dict(extra_detection_noise_px=_noise,
                    kalman=round(float(np.nanmean(_kk)), 3),
                    raw_single_frame=round(float(np.nanmean(_rr)), 3)))
_SW = pd.DataFrame(_sw).set_index("extra_detection_noise_px")
_SW["kalman_advantage"] = (_SW.kalman - _SW.raw_single_frame).round(3)
display(_SW.style.background_gradient(subset=["kalman_advantage"], cmap="RdYlGn")
        .set_caption("Filtering helps only once localisation noise is large "
                     "relative to inter-frame motion"))
print("When detections are clean relative to how far a fish moves between frames, the raw")
print("difference is already a good heading estimate and the filter only adds lag. The")
print("filter earns its place in the regime this table shows, and the rig determines which")
print("regime you are in -- which is a camera and frame-rate decision, not a code decision.")

print("\nIMPORTANT CAVEAT on the absolute values: measured polarisation (~"
      f"{_rep.polarisation_kalman.max():.2f}) sits well below the injected "
      f"{_rep.true_polarisation.max():.2f}.")
print("Swim jitter and localisation noise both push the estimate down, so this indicator is")
print("ORDINAL, not calibrated. Use it as a trend against a site- and stock-specific")
print("baseline. A raw polarisation number compared against a literature threshold would be")
print("meaningless, and the welfare index in Part 7 treats it accordingly.")
_bad = TRK[~TRK.behaviour_reportable]
if len(_bad):
    print(f"\n{len(_bad)} state(s) fall below the purity gate and are marked NOT REPORTABLE:")
    for nm, r_ in _bad.iterrows():
        print(f"  '{nm}': purity {r_.mean_track_purity:.2f}, "
              f"{r_.tracks_created:.0f} tracks for {r_.true_fish:.0f} fish "
              f"({r_.count_err_pct:+.0f}% count error), polarisation reads "
              f"{r_.polarisation_kalman:.2f} against a true {r_.true_polarisation:.2f}")
    print("It reads HIGHER than the calmer states, i.e. the indicator inverts. A fragmenting")
    print("tracker produces short tracks whose velocity estimates are dominated by the")
    print("initialisation, and those look spuriously aligned. Reporting that number would")
    print("say the stressed school is CALMER than the calm one.")
    print("So the welfare index in Part 7 must consume behaviour indicators ONLY when purity")
    print("clears the gate, and mark them unavailable otherwise -- never substitute a wrong")
    print("value for a missing one.")
print(f"[VET] track purity stays above {_ok.mean_track_purity.min():.2f} on calm and moderate")
print(f"      schools, and degrades to {TRK.mean_track_purity.min():.2f} on the erratic one")
print("      -> PASS (the degradation is the finding, not a failure)")
print("\nNote the count error rising as the school gets erratic: the tracker fragments, and")
print("fragmentation INFLATES counts. An abundance or stocking figure taken from track counts")
print("during a stress event is biased high exactly when it matters. Report purity alongside.")

dets, gt, _ = simulate_swim(polarisation=0.60, speed_BL=1.4, seed=RUN["SEED"]+1)
A, ntr, tracks = run_tracker(dets)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for i in range(gt.shape[1]): ax[0].plot(gt[:, i, 0], gt[:, i, 1], lw=.8, alpha=.7)
ax[0].set_xlim(0, ARENA_W); ax[0].set_ylim(ARENA_H, 0)
ax[0].set_title("true trajectories (tracking arena)")
V = np.diff(gt, axis=0); U = V/np.maximum(np.linalg.norm(V, axis=-1, keepdims=True), 1e-9)
phi_t = np.linalg.norm(U.mean(axis=1), axis=-1)
ax[1].plot(phi_t, color="#1F3A5F"); ax[1].axhline(0.35, color="crimson", ls="--",
                                                   label="welfare band lower limit")
ax[1].set_xlabel("frame"); ax[1].set_ylabel("polarisation")
ax[1].legend(fontsize=7); ax[1].set_title("Schooling polarisation over time")
plt.tight_layout(); plt.show()

## Part 9 — Bioenergetics: closing the loop with the thermal model

This is where the two notebooks become one model. The thermal notebook delivers a captured
heat stream at a temperature; that temperature drives growth, oxygen demand and ammonia
production; those in turn set the **stocking density the loop can actually carry**.

### 9.1 Growth

Two standard indices, both reported because they answer different questions.

**Specific growth rate** — relative, and it falls with body size:

$$\mathrm{SGR} = \frac{\ln W_t - \ln W_0}{t}\times 100 \ \ (\%\,\mathrm{d}^{-1})$$

**Thermal growth coefficient** — normalised for temperature via degree-days, so growth at
different temperatures is comparable (Iwama & Tautz 1981; Cho 1992):

$$\mathrm{TGC} = \frac{W_t^{1/3}-W_0^{1/3}}{\sum_t T_t}\times 1000
\qquad\Longrightarrow\qquad
W_t = \left(W_0^{1/3} + \tfrac{\mathrm{TGC}}{1000}\sum_t T_t\right)^{3}$$

Reported Atlantic salmon values span roughly **0.25 (remedial) to 0.35 (elite)** on this
×1000 convention.

**The cautionary note matters here.** Jobling (2003) is explicit that TGC assumes growth is
linear in temperature, whereas the true growth–temperature relationship is **bell-shaped**.
Extrapolating TGC above the thermal optimum **overestimates growth** — which is precisely the
direction that would make a thermal-reuse proposal look good. So we multiply the TGC
increment by a thermal performance curve and show both, side by side.

### 9.2 Loading

Oxygen demand and ammonia production scale with feed and with temperature. Published RAS
benchmarks: routine oxygen consumption on the order of **150–230 mg O₂ kg⁻¹ h⁻¹** with strong
post-prandial peaks, and TAN excretion of roughly **4.6–10.8 mg TAN kg⁻¹ h⁻¹**, driven mainly
by dietary protein. From the nitrogen balance, TAN is about **37–41 g per kg of feed** at
36–40 % crude protein (N ≈ 16 % of protein).

Then the toxic fraction follows Emerson et al. (1975), and it **rises with temperature**:

$$pK_a = 0.09018 + \frac{2729.92}{T+273.15},\qquad f_{NH_3} = \frac{1}{1+10^{\,pK_a-\mathrm{pH}}}$$

So warming the loop raises growth **and** raises oxygen demand **and** lowers oxygen solubility
**and** raises ammonia toxicity. The stocking ceiling is where those four curves cross.

In [ ]:
# ---- oxygen solubility and ammonia speciation (same functions as the thermal notebook) ----
def rho_sw(T, S):
    T = np.asarray(T, float); S = np.asarray(S, float)
    rw = (999.842594 + 6.793952e-2*T - 9.095290e-3*T**2 + 1.001685e-4*T**3
          - 1.120083e-6*T**4 + 6.536332e-9*T**5)
    A = 8.24493e-1 - 4.0899e-3*T + 7.6438e-5*T**2 - 8.2467e-7*T**3 + 5.3875e-9*T**4
    Bc = -5.72466e-3 + 1.0227e-4*T - 1.6546e-6*T**2
    return rw + A*S + Bc*S**1.5 + 4.8314e-4*S**2

_GG = dict(A0=5.80871, A1=3.20291, A2=4.17887, A3=5.10006, A4=-9.86643e-2, A5=3.80369,
           B0=-7.01577e-3, B1=-7.70028e-3, B2=-1.13864e-2, B3=-9.51519e-3, C0=-2.75915e-7)
def o2_sat_mgL(T, S):
    T = np.asarray(T, float); S = np.asarray(S, float)
    Ts = np.log((298.15-T)/(273.15+T)); g = _GG
    ln = (g["A0"]+g["A1"]*Ts+g["A2"]*Ts**2+g["A3"]*Ts**3+g["A4"]*Ts**4+g["A5"]*Ts**5
          + S*(g["B0"]+g["B1"]*Ts+g["B2"]*Ts**2+g["B3"]*Ts**3) + g["C0"]*S**2)
    return np.exp(ln)*31.9988e-3*rho_sw(T, S)/1000.0

def f_nh3(T, pH, S=30.5):
    pKa_fw = 0.09018 + 2729.92/(np.asarray(T, float)+273.15)
    I = 19.9273*S/(1000-1.005109*S)
    pKa = pKa_fw - (0.1552*np.sqrt(I) - 0.0592*I)
    return 1.0/(1.0+10**(pKa-pH))

# ---- VET against published reference values --------------------------------
_e = abs(float(o2_sat_mgL(10, 33)) - 9.26)
assert _e < 0.25, "O2 solubility is off"
_f20 = 1/(1+10**((0.09018+2729.92/(20+273.15))-8.0))
assert abs(100*_f20 - 3.83) < 0.1, "Emerson NH3 is off"
print(f"[VET] O2 at 10 C, S=33 -> {float(o2_sat_mgL(10,33)):.2f} mg/L (literature ~9.2-9.3)")
print(f"[VET] NH3 fraction at 20 C, pH 8.0, fresh -> {100*_f20:.2f} % (published 3.83 %) -> PASS")

SPECIES = dict(name="salmonid-like (PLACEHOLDER)", T_min=4.0, T_opt=15.0, T_max=25.1,
               TGC=0.30, FCR=1.15, protein_pct=0.40,
               O2_mg_per_kg_h_at_Topt=190.0, Q10=2.0,
               DO_good=7.0, DO_crit=5.0, NH3_caution=0.012, NH3_danger=0.050,
               source="EPA R10 / Brett 1952 / FAO TGC ranges / RAS benchmarks; SET per species")

def tpc_beta(T, Tmin, Topt, Tmax):
    T = np.asarray(T, float); out = np.zeros_like(T)
    m = (T > Tmin) & (T < Tmax); e = (Tmax-Topt)/(Topt-Tmin)
    out[m] = ((T[m]-Tmin)*(Tmax-T[m])**e)/((Topt-Tmin)*(Tmax-Topt)**e)
    return np.clip(out, 0, 1)

def grow(W0_g, T_C, days, TGC=None, thermal_limited=True, sp=SPECIES):
    TGC = sp["TGC"] if TGC is None else TGC
    T = np.full(days, float(T_C))
    inc = TGC/1000.0*T
    if thermal_limited:
        inc = inc*tpc_beta(T, sp["T_min"], sp["T_opt"], sp["T_max"])
    W = (W0_g**(1/3) + np.cumsum(inc))**3
    return W

def sgr(W0, Wt, days): return 100*(np.log(Wt)-np.log(W0))/days

# ---- VET: TGC inverts exactly (constant T, no thermal limiter) --------------
_W0, _T, _d, _tgc = 150.0, 14.0, 120, 0.30
_Wt = float(grow(_W0, _T, _d, _tgc, thermal_limited=False)[-1])
_rec = (_Wt**(1/3) - _W0**(1/3))/(_T*_d)*1000
assert abs(_rec - _tgc) < 1e-9, "TGC forward/inverse is inconsistent"
print(f"[VET] TGC round trip: grew {_W0} g -> {_Wt:.1f} g over {_d} d at {_T} C; "
      f"recovered TGC = {_rec:.6f} (input {_tgc}) -> PASS")

TT = np.linspace(6, 24, 80)
naive = np.array([grow(150., t, 180, thermal_limited=False)[-1] for t in TT])
limited = np.array([grow(150., t, 180, thermal_limited=True)[-1] for t in TT])
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.2))
ax[0].plot(TT, naive/1000, "--", color="#C0392B", label="raw TGC (linear in T)")
ax[0].plot(TT, limited/1000, color="#1F3A5F", label="TGC x thermal performance")
ax[0].axvline(AQ["loop_inlet_T_C"], color="k", ls=":", lw=1)
ax[0].annotate(f"loop {AQ['loop_inlet_T_C']:.1f} C", (AQ["loop_inlet_T_C"]+.2, .5), fontsize=7)
ax[0].set_xlabel("constant temperature (C)"); ax[0].set_ylabel("mass after 180 d (kg)")
ax[0].legend(fontsize=7); ax[0].set_title("The Jobling (2003) warning, drawn")
d = np.arange(1, 241)
for t, c in [(11.5, "#7FA6C9"), (AQ["loop_inlet_T_C"], "#1F3A5F"), (22.0, "#C0392B")]:
    ax[1].plot(d, grow(150., t, 240)/1000, color=c, label=f"{t:.1f} C")
ax[1].set_xlabel("days"); ax[1].set_ylabel("mass (kg)"); ax[1].legend(fontsize=7)
ax[1].set_title("Growth trajectories")
ax[2].plot(TT, [sgr(150., grow(150., t, 180)[-1], 180) for t in TT], color="#2E7D82")
ax[2].set_xlabel("temperature (C)"); ax[2].set_ylabel("SGR (%/day)")
ax[2].set_title("Specific growth rate")
plt.tight_layout(); plt.show()
opt_i = int(np.argmax(limited))
print(f"Growth peaks at {TT[opt_i]:.1f} C. Above that the raw TGC keeps rising while the")
print(f"thermally-limited curve falls: at 22 C the naive model overstates final mass by "
      f"{100*(naive[np.argmin(abs(TT-22))]/limited[np.argmin(abs(TT-22))]-1):.0f}%.")
print("A reuse proposal built on raw TGC would claim a benefit that the biology does not give.")

In [ ]:
def loading(biomass_kg, T_C, S=30.5, pH=7.95, sp=SPECIES, feed_rate_pct=1.2,
            exchange_m3_per_h=None, tank_m3=200.0):
    '''Oxygen demand, TAN production and the resulting steady-state DO and NH3.'''
    q10 = sp["Q10"]**((T_C-sp["T_opt"])/10.0)
    O2_mg_h = biomass_kg*sp["O2_mg_per_kg_h_at_Topt"]*q10
    feed_kg_d = biomass_kg*feed_rate_pct/100.0
    TAN_g_per_kg_feed = 1000*sp["protein_pct"]*0.16*0.60     # 60% of ingested N excreted as TAN
    TAN_g_h = feed_kg_d*TAN_g_per_kg_feed/24.0
    sat = float(o2_sat_mgL(T_C, S))
    Q = exchange_m3_per_h if exchange_m3_per_h else tank_m3*0.5
    DO_out = sat - O2_mg_h/max(Q*1000.0, 1e-9)               # mg/L drawdown across the tank
    TAN_mgL = TAN_g_h*1000.0/max(Q*1000.0, 1e-9)
    NH3_mgL = TAN_mgL*float(f_nh3(T_C, pH, S))
    return dict(O2_demand_g_h=O2_mg_h/1000, feed_kg_d=feed_kg_d,
                TAN_g_h=TAN_g_h, O2_sat_mgL=sat, DO_out_mgL=DO_out,
                TAN_mgL=TAN_mgL, NH3_mgL=NH3_mgL,
                TAN_g_per_kg_feed=TAN_g_per_kg_feed)

_chk = loading(1000., 15.0)
print(f"[VET] TAN yield {_chk['TAN_g_per_kg_feed']:.1f} g per kg feed at 40% protein "
      f"(published 36.7-40.6 g/kg at 36-40% CP)")
print(f"[VET] O2 demand {1000*_chk['O2_demand_g_h']/1000:.0f} mg/kg/h at T_opt "
      f"(RAS benchmarks 146-226 mg/kg/h) -> in range")

# ---- stocking ceiling: where does each constraint bite? --------------------
dens = np.linspace(2, 60, 120)                 # kg/m3
TANK_M3, EXCH = 200.0, 100.0
rows = []
for T in [11.5, AQ["loop_inlet_T_C"], 18.0, 21.0]:
    lim = {}
    for d in dens:
        L = loading(d*TANK_M3, T, exchange_m3_per_h=EXCH, tank_m3=TANK_M3)
        if "DO" not in lim and L["DO_out_mgL"] < SPECIES["DO_crit"]: lim["DO"] = d
        if "NH3" not in lim and L["NH3_mgL"] > SPECIES["NH3_caution"]: lim["NH3"] = d
    rows.append(dict(T_C=T, DO_limit_kg_m3=lim.get("DO", np.nan),
                     NH3_limit_kg_m3=lim.get("NH3", np.nan),
                     binding=("DO" if lim.get("DO", 1e9) < lim.get("NH3", 1e9) else "NH3"),
                     ceiling_kg_m3=min(lim.get("DO", np.nan), lim.get("NH3", np.nan)),
                     growth_rel=float(tpc_beta(np.array([T]), SPECIES["T_min"],
                                               SPECIES["T_opt"], SPECIES["T_max"])[0])))
CEIL = pd.DataFrame(rows).set_index("T_C")
display(CEIL.round(2).style.set_caption(
    f"Stocking ceiling, {TANK_M3:.0f} m3 tank at {EXCH:.0f} m3/h exchange (PLACEHOLDER limits)"))

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.2))
for T, c in zip([11.5, AQ["loop_inlet_T_C"], 21.0], ["#7FA6C9", "#1F3A5F", "#C0392B"]):
    do = [loading(d*TANK_M3, T, exchange_m3_per_h=EXCH, tank_m3=TANK_M3)["DO_out_mgL"] for d in dens]
    nh = [loading(d*TANK_M3, T, exchange_m3_per_h=EXCH, tank_m3=TANK_M3)["NH3_mgL"] for d in dens]
    ax[0].plot(dens, do, color=c, label=f"{T:.1f} C")
    ax[1].plot(dens, nh, color=c, label=f"{T:.1f} C")
ax[0].axhline(SPECIES["DO_crit"], color="crimson", ls="--", label="DO critical")
ax[0].set_xlabel("stocking density (kg/m3)"); ax[0].set_ylabel("tank-outlet DO (mg/L)")
ax[0].legend(fontsize=6.5); ax[0].set_title("Oxygen")
ax[1].axhline(SPECIES["NH3_caution"], color="crimson", ls="--", label="NH3 caution")
ax[1].set_xlabel("stocking density (kg/m3)"); ax[1].set_ylabel("un-ionised NH3 (mg/L)")
ax[1].legend(fontsize=6.5); ax[1].set_title("Ammonia")
TT2 = np.linspace(9, 23, 40)
ceil, growth = [], []
for T in TT2:
    c_ = np.nan
    for d in dens:
        L = loading(d*TANK_M3, T, exchange_m3_per_h=EXCH, tank_m3=TANK_M3)
        if L["DO_out_mgL"] < SPECIES["DO_crit"] or L["NH3_mgL"] > SPECIES["NH3_caution"]:
            c_ = d; break
    ceil.append(c_); growth.append(float(tpc_beta(np.array([T]), SPECIES["T_min"],
                                                  SPECIES["T_opt"], SPECIES["T_max"])[0]))
ceil = np.array(ceil); growth = np.array(growth)
ax[2].plot(TT2, ceil, color="#1F3A5F", label="stocking ceiling (kg/m3)")
a2 = ax[2].twinx(); a2.plot(TT2, ceil*growth, color="#C0392B", label="ceiling x growth = yield proxy")
a2.set_ylabel("yield proxy", color="#C0392B")
ax[2].axvline(AQ["loop_inlet_T_C"], color="k", ls=":")
ax[2].set_xlabel("loop temperature (C)"); ax[2].set_ylabel("kg/m3", color="#1F3A5F")
ax[2].set_title("The optimum is not the growth optimum")
plt.tight_layout(); plt.show()

best = TT2[int(np.nanargmax(ceil*growth))]
print(f"\nGrowth alone peaks near {SPECIES['T_opt']:.0f} C, but the stocking ceiling falls with")
print(f"temperature, so the YIELD optimum sits at about {best:.1f} C.")
print("Heating the loop past that point buys faster individual growth and loses more in")
print("carrying capacity than it gains. That is the quantitative answer to 'how much of the")
print("captured heat should we actually use', and it is the number to send back to the")
print("thermal notebook as an upper bound on the reuse loop's inlet temperature.")
print("\nEvery limit here is a PLACEHOLDER. The structure is the deliverable; the numbers come")
print("from the species, the life stage, the diet and a veterinary review.")

## Part 10 — Evaluation, power analysis, and what the system can honestly claim

### 10.1 How many fish must you measure?

The standard error of mean mass over $n$ measured fish is $\sigma_W/\sqrt{n}$, so the
half-width of a 95 % interval on **total biomass** is

$$\frac{w}{\bar W} \approx \frac{1.96}{\sqrt{n}}\cdot\mathrm{CV}(W)$$

Biological CV alone is typically 0.25–0.40 for a graded cohort, before any measurement error.
This sets the sampling effort — and it is usually larger than people expect.

In [ ]:
def n_for_halfwidth(cv, w_rel, z=1.96): return int(np.ceil((z*cv/w_rel)**2))
rows = []
for cv in [0.20, 0.30, 0.40]:
    r = dict(CV_of_mass=cv)
    for w in [0.10, 0.05, 0.02]:
        r[f"n_for_+-{int(100*w)}%"] = n_for_halfwidth(cv, w)
    rows.append(r)
display(pd.DataFrame(rows).set_index("CV_of_mass")
        .style.set_caption("Fish that must be MEASURED for a given CI on mean mass"))
cv_obs = float(np.std(mass_from_length(pop_meas_L))/np.mean(mass_from_length(pop_meas_L)))
print(f"observed CV of estimated mass in this run: {cv_obs:.3f}")
print(f"-> for +/-5% on mean mass you need about {n_for_halfwidth(cv_obs, 0.05)} measured fish,")
print(f"   and you measured {len(pop_meas_L)}.")
print("Detection probability (Part 6) means the number of fish you must SHOW the camera is")
print(f"roughly {n_for_halfwidth(cv_obs,0.05)/max(np.mean(pi_of_L(pop_true_L)),1e-6):.0f}.")

# ---- end-to-end scorecard --------------------------------------------------
seg_iou = np.mean([iou for _, _, iou in pairs]) if pairs else np.nan
SCORE = pd.DataFrame([
 ("instance count", f"{len(INST)} detected vs {len(TRUTH)} true",
  f"{100*(len(INST)/max(len(TRUTH),1)-1):+.0f}%", "watershed splitting; see Part 4 VET"),
 ("segmentation IoU (matched)", f"{seg_iou:.3f}", "-", "classical baseline; deep model would improve"),
 ("length bias", f"{M.L_err_pct.mean():+.2f}%", "-", "curvilinear estimator, true range supplied"),
 ("length MAPE", f"{M.L_err_pct.abs().mean():.2f}%", "-", "per fish"),
 ("mass bias", f"{M.mass_err_pct.mean():+.2f}%", "-", "includes biological L-W scatter"),
 ("biomass error (all corrections)", f"{BIO.loc['all corrections','biomass_err_pct']:+.2f}%",
  "-", "population level, the estimable quantity"),
 ("biomass error (no corrections)", f"{BIO.loc['raw (no corrections)','biomass_err_pct']:+.2f}%",
  "-", "what you get by skipping Part 6"),
 ("welfare index", f"{score:.3f} ({grade(score)})", f"{100*cov:.0f}% weight coverage",
  "PARTIAL — see the NOT COVERED rows"),
], columns=["metric", "value", "vs truth", "note"])
display(SCORE.style.set_caption("End-to-end scorecard against exact ground truth"))

# ---- robustness sweep -------------------------------------------------------
rows = []
for nm, kw in [("baseline", {}), ("turbid x2", dict(turbid=2.0)),
               ("crowded (24 fish)", dict(n_fish=24)),
               ("high bend (85 deg)", dict(bend_max_deg=85)),
               ("distant (200-320 cm)", dict(Z_range=(200, 320)))]:
    im, tr = render_tank(seed=RUN["SEED"]+7, **{**dict(n_fish=16), **kw})
    lab, n = split_instances(foreground(im), min_sep=13, peak_frac=0.42, min_area=110)
    ins = measure_instances(lab, im)
    pr = match_to_truth(ins, tr)
    Zs = np.full(max(lab.max(), 1), 150.0)
    for a, b_, _ in pr: Zs[a] = tr[b_]["Z_cm"]
    ins = measure_instances(lab, im, Z_known=Zs); pr = match_to_truth(ins, tr)
    if pr:
        le = np.array([100*(ins[a]["L_cm"]/tr[b_]["L_cm"]-1) for a, b_, _ in pr])
        rows.append(dict(condition=nm, true_fish=len(tr), detected=n,
                         matched=len(pr), recall=len(pr)/max(len(tr), 1),
                         mean_IoU=np.mean([i for _, _, i in pr]),
                         L_bias_pct=le.mean(), L_MAPE_pct=np.abs(le).mean()))
    else:
        rows.append(dict(condition=nm, true_fish=len(tr), detected=n, matched=0,
                         recall=0.0, mean_IoU=np.nan, L_bias_pct=np.nan, L_MAPE_pct=np.nan))
ROB = pd.DataFrame(rows).set_index("condition")
display(ROB.round(3).style.background_gradient(subset=["recall"], cmap="RdYlGn")
        .set_caption("Robustness — where the measurement chain degrades"))
print("Turbidity and range hit RECALL (the size-selectivity problem of Part 6). Crowding hits")
print("the SPLITTING. Bend hits the LENGTH estimator. Three different failures needing three")
print("different fixes -- which is why they are reported separately.")

## Part 11 — Deep instance segmentation (Colab GPU, guarded)

The classical baseline separates touching fish when the necks are narrow. It fails on heavy
overlap and on fish partly behind equipment. A learned instance-segmentation model fixes that,
and **everything downstream in this notebook is mask-agnostic**: feed it predicted masks and
the morphometrics, biomass corrections, welfare index and bioenergetics run unchanged.

Licence note: Mask R-CNN via torchvision is BSD-licensed; **Ultralytics YOLO is AGPL-3.0**,
which is copyleft and has real implications if anything here is distributed. Record the
choice and its licence in the repo.

In [ ]:
def train_maskrcnn(data_dir, epochs=8, lr=5e-3):
    '''Fine-tune torchvision Mask R-CNN for single-class fish instance segmentation.
    Split by TANK/SESSION, never by frame. Run in Colab with a GPU.'''
    import torch
    from torchvision.models.detection import maskrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = maskrcnn_resnet50_fpn(weights="DEFAULT")
    inf = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(inf, 2)         # background + fish
    inm = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(inm, 256, 2)
    model.to(dev)
    print("Prepare a COCO-style dataset, then run the standard torchvision detection loop.")
    print("Evaluation to report: mask AP@[.5:.95], AND the length/mass bias from Part 5 --")
    print("a higher AP with a worse length bias is a WORSE measurement system.")
    return model

try:
    import torch; HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
print(f"PyTorch available: {HAVE_TORCH} | RUN_REAL_DATA: {RUN['RUN_REAL_DATA']}")
if HAVE_TORCH and RUN["RUN_REAL_DATA"]:
    print("Call train_maskrcnn('<data_dir>').")
else:
    print("Deep path skipped. The classical baseline carried the whole notebook, which is the")
    print("point: the measurement science is separable from the segmentation model, and the")
    print("bias corrections in Part 6 matter more than the architecture.")

## Part 12 — Safety rules, handoff, and the conditional finding

In [ ]:
RULES = [
 dict(id="W1_welfare_poor", severity=5, action="escalate_to_vet_and_stop_heat_input",
      trip=dict(metric="welfare_index", op=">", value=0.50), clear=dict(value=0.38),
      debounce_n=2, basis="partial welfare index in the 'poor' band"),
 dict(id="W2_welfare_marginal", severity=3, action="queue_human_review",
      trip=dict(metric="welfare_index", op=">", value=0.30), clear=dict(value=0.22),
      debounce_n=4, basis="'marginal' band"),
 dict(id="W3_do_floor", severity=5, action="stop_heat_input_and_aerate",
      trip=dict(metric="DO_mgL", op="<", value=5.0), clear=dict(value=6.2),
      debounce_n=2, basis="species hypoxia threshold; SET with advice"),
 dict(id="W4_nh3", severity=4, action="stop_feeding_and_increase_exchange",
      trip=dict(metric="NH3_mgL", op=">", value=0.012), clear=dict(value=0.008),
      debounce_n=3, basis="common aquaculture caution level"),
 dict(id="W5_loop_temp_ceiling", severity=4, action="reduce_heat_capture",
      trip=dict(metric="loop_T_C", op=">", value=float(min(19.0, SPECIES['T_opt']+3))),
      clear=dict(value=float(SPECIES['T_opt']+1)), debounce_n=3,
      basis="yield-optimum ceiling from Part 9, NOT the growth optimum"),
 dict(id="W6_stocking", severity=4, action="halt_stocking_and_review",
      trip=dict(metric="density_kg_m3", op=">", value=float(np.nanmin(CEIL.ceiling_kg_m3))),
      clear=dict(value=float(0.8*np.nanmin(CEIL.ceiling_kg_m3))), debounce_n=1,
      basis="the binding DO/NH3 ceiling at the current loop temperature"),
 dict(id="W7_biomass_CI_too_wide", severity=2, action="measure_more_fish",
      trip=dict(metric="biomass_CI_halfwidth_pct", op=">", value=10.0),
      clear=dict(value=6.0), debounce_n=1,
      basis="a density decision on a +/-10% estimate is not a decision"),
 dict(id="W8_stale_or_blind", severity=5, action="failsafe_escalate",
      trip=dict(metric="data_age_s", op=">", value=900), clear=dict(value=300),
      debounce_n=1, basis="missing or stale input must never read as 'all fine'"),
]

class RuleEngine:
    def __init__(self, rules):
        self.rules = rules; self.state = {r["id"]: dict(active=False, count=0) for r in rules}
    def _c(self, v, op, x): return v > x if op == ">" else v < x
    def step(self, obs):
        out = []
        for r in self.rules:
            st = self.state[r["id"]]; m = r["trip"]["metric"]; v = obs.get(m)
            if v is None or (isinstance(v, float) and not np.isfinite(v)):
                out.append(dict(id=r["id"], action="failsafe_escalate", severity=5,
                                reason=f"{m} missing")); st["active"] = True; continue
            trip = self._c(v, r["trip"]["op"], r["trip"]["value"])
            clear = not self._c(v, r["trip"]["op"], r["clear"]["value"])
            if not st["active"]:
                st["count"] = st["count"]+1 if trip else 0
                if st["count"] >= r["debounce_n"]:
                    st["active"] = True
                    out.append(dict(id=r["id"], action=r["action"], severity=r["severity"],
                                    reason=f"{m}={v:.4g} {r['trip']['op']} {r['trip']['value']:.4g}"))
            elif clear:
                st["count"] = 0; st["active"] = False
        return sorted(out, key=lambda d: -d["severity"])

eng = RuleEngine(RULES)
hrs = np.arange(0, 72, 0.5)
Tloop_t = AQ["loop_inlet_T_C"] + 2.6*np.sin(2*np.pi*hrs/24) + 0.9*(hrs > 48)
DO_t = np.array([float(o2_sat_mgL(t, 30.5))*0.78 for t in Tloop_t])
NH3_t = 0.006*2.0**((Tloop_t-AQ["loop_inlet_T_C"])/12.0)
wel_t = np.array([welfare_index(IND, GROUP, dict(ENV, T_C=t, DO_mgL=d, NH3_mgL=n))[0]
                  for t, d, n in zip(Tloop_t, DO_t, NH3_t)])
log = []
for i, h in enumerate(hrs):
    obs = dict(welfare_index=float(wel_t[i]), DO_mgL=float(DO_t[i]), NH3_mgL=float(NH3_t[i]),
               loop_T_C=float(Tloop_t[i]), density_kg_m3=18.0,
               biomass_CI_halfwidth_pct=float(100*(hi-lo)/2/mid),
               data_age_s=120 if i != 70 else 1800)
    for f in eng.step(obs): log.append(dict(hour=round(h, 1), **f))
LOG = pd.DataFrame(log)
print(f"{len(LOG)} rule activations over 72 h")
if len(LOG):
    display(LOG.groupby(["id", "action", "severity"]).size().rename("n").reset_index())

fig, ax = plt.subplots(3, 1, figsize=(11, 5.4), sharex=True)
ax[0].plot(hrs, Tloop_t, color="#C0392B"); ax[0].axhline(SPECIES["T_opt"], ls=":", color="k")
ax[0].set_ylabel("loop T (C)"); ax[0].set_title("Simulated 72 h of operation")
ax[1].plot(hrs, DO_t, color="#1F3A5F", label="DO")
ax[1].axhline(SPECIES["DO_crit"], color="crimson", ls="--", label="critical")
ax[1].set_ylabel("DO (mg/L)"); ax[1].legend(fontsize=7)
ax[2].plot(hrs, wel_t, color="#2E7D82")
ax[2].axhline(.30, color="#8E6C1F", ls="--", label="marginal"); ax[2].axhline(.50, color="crimson", ls="--", label="poor")
ax[2].fill_between(hrs, .22, .30, color="#8E6C1F", alpha=.10)
ax[2].set_ylabel("partial welfare index"); ax[2].set_xlabel("hours"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("The shaded band is the hysteresis gap on W2: the index must fall to 0.22 to clear a")
print("trip at 0.30, so a diurnal cycle brushing the threshold cannot chatter the alarm.")

In [ ]:
# ============================= HANDOFF ARTIFACTS ==============================
E = RUN["EXPORT_DIR"]; os.makedirs(E, exist_ok=True)
M.to_csv(f"{E}/per_fish_measurements.csv", index=False)
MORPH.to_csv(f"{E}/length_estimator_vetting.csv")
BIO.to_csv(f"{E}/biomass_corrections.csv")
JB.to_csv(f"{E}/jensen_corrections.csv")
OWI.to_csv(f"{E}/welfare_indicator_weights.csv", index=False)
BR.to_csv(f"{E}/welfare_breakdown.csv", index=False)
TRK.to_csv(f"{E}/tracking_metrics.csv")
CEIL.to_csv(f"{E}/stocking_ceiling.csv")
ROB.to_csv(f"{E}/robustness.csv")
SCORE.to_csv(f"{E}/scorecard.csv", index=False)
BUD.to_csv(f"{E}/optical_error_budget.csv")

AQUA_HANDOFF = dict(
  meta=dict(notebook="Aquatic Guardian - Mentee 2 - aquaculture vision", seed=RUN["SEED"],
            fast_mode=RUN["FAST"],
            generated_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())),
  safety_contract=SAFETY_CONTRACT,
  consumed_from_thermal=AQ,
  verification=dict(renderer_projection="PASS", refraction_inverse="PASS",
                    watershed_instance_count="PASS", curvilinear_length_bias="PASS",
                    jensen_corrections="PASS", o2_solubility="PASS", nh3_emerson="PASS",
                    tgc_roundtrip="PASS", tracker_polarisation="PASS"),
  optics=dict(f_px=CAM["f_px"], baseline_cm=CAM["baseline_cm"], n_water=N_WATER,
              refraction_error_if_ignored_pct=34.0,
              mass_error_if_refraction_ignored_pct=140.0),
  measurement=dict(length_bias_pct=float(M.L_err_pct.mean()),
                   length_MAPE_pct=float(M.L_err_pct.abs().mean()),
                   chord_bias_pct=float(M.L_chord_err_pct.mean()),
                   mass_MAPE_pct=float(M.mass_err_pct.abs().mean()),
                   segmentation_IoU=float(seg_iou)),
  biomass=dict(total_kg=float(mid), ci95_kg=[float(lo), float(hi)],
               ci_halfwidth_pct=float(100*(hi-lo)/2/mid),
               true_kg=float(truth_total/1000),
               error_all_corrections_pct=float(BIO.loc["all corrections", "biomass_err_pct"]),
               error_no_corrections_pct=float(BIO.loc["raw (no corrections)", "biomass_err_pct"]),
               corrections_applied=["log-normal smearing exp(s^2/2)",
                                    "Horvitz-Thompson size-selectivity weights",
                                    "curvilinear length (not chord)"],
               n_measured=int(len(pop_meas_L)),
               n_needed_for_5pct=int(n_for_halfwidth(cv_obs, 0.05))),
  welfare=dict(partial_index=float(score), grade=grade(score),
               weight_coverage_pct=float(100*cov),
               vision_only_coverage_pct=float(100*vis_only),
               not_covered=OWI[OWI.source == "NOT COVERED"].indicator.tolist(),
               bands_are="PLACEHOLDERS pending veterinary and aquatic-science review"),
  # ---- THE NUMBER THE THERMAL NOTEBOOK NEEDS BACK ----
  reuse_constraint=dict(
    growth_optimum_C=float(TT[opt_i]),
    yield_optimum_C=float(best),
    welfare_marginal_at_C=float(T_lim) if np.isfinite(T_lim) else None,
    recommended_loop_inlet_ceiling_C=float(min(best, T_lim if np.isfinite(T_lim) else best)),
    stocking_ceiling_kg_m3_at_loop_T=float(CEIL.ceiling_kg_m3.loc[AQ["loop_inlet_T_C"]])
        if AQ["loop_inlet_T_C"] in CEIL.index else float(np.nanmin(CEIL.ceiling_kg_m3)),
    binding_constraint=str(CEIL.binding.iloc[1]),
    note="Heating the loop above the YIELD optimum loses more carrying capacity than it "
         "gains in growth. This ceiling bounds how much captured heat the loop can absorb."),
  safety_rules=RULES,
  open_gaps=["no real-data validation in this run" if not RUN["RUN_REAL_DATA"] else "real data used",
             "stereo range assumed known; a monocular rig cannot supply it",
             "fin, skin, gill, mortality and appetite indicators NOT covered by vision",
             "all welfare bands, TGC, FCR and loading constants are placeholders",
             "no biological validation experiment has been run"],
)
with open(f"{E}/aquaculture_handoff.json", "w") as f:
    json.dump(AQUA_HANDOFF, f, indent=2, default=str)
print("Exported:")
for f in sorted(os.listdir(E)):
    if f.endswith((".csv", ".json")) and "aquaculture" in f or f.endswith(".csv"):
        print("  ", f)
print("\naquaculture_handoff.json closes the loop with thermal_handoff.json:")
print(f"  thermal ->  loop inlet {AQ['loop_inlet_T_C']:.1f} C, DO {AQ['DO_at_loop_T_mgL']:.2f} mg/L")
print(f"  aqua    ->  recommended inlet CEILING "
      f"{AQUA_HANDOFF['reuse_constraint']['recommended_loop_inlet_ceiling_C']:.1f} C, "
      f"stocking ceiling {AQUA_HANDOFF['reuse_constraint']['stocking_ceiling_kg_m3_at_loop_T']:.0f} kg/m3")

### 12.1 Conditional finding — the template to complete

> Measuring a rendered stock with exact ground truth, the pipeline recovers individual length
> with a bias of **{L_bias}%** and MAPE **{L_mape}%**, and population biomass to within
> **{bio_err}%** once the log-normal smearing and size-selectivity corrections are applied —
> against **{bio_raw}%** without them. Total biomass is **{kg} kg (95 % CI {ci})**, an interval
> half-width of **{hw}%**, which {is/is not} tight enough to support a stocking-density
> decision at the **{ceiling} kg/m³** ceiling.
>
> The partial welfare index is **{w} ({grade})**, covering **{cov}%** of the SWIM-style
> indicator weight. **Fin, skin and gill condition, mortality rate and appetite are not
> covered by vision** and must come from husbandry records and handling assessments.
>
> Growth peaks at **{Topt} °C**, but the DO- and ammonia-limited stocking ceiling falls with
> temperature, so the yield optimum is **{Tyield} °C** and the welfare index reaches
> 'marginal' at **{Tmarg} °C**. The recommended ceiling on the reuse-loop inlet temperature is
> therefore **{Tceil} °C**, which bounds the heat the loop can absorb and is returned to the
> thermal model as a constraint.
>
> **This is measurement and triage evidence.** It does not establish that the loop is
> operating safely, it does not diagnose any animal, and it does not authorise a thermal-reuse
> decision. Every threshold is a placeholder pending veterinary and aquatic-science review.

### 12.2 Out of scope

Any release of cultured organisms to the wild; disease diagnosis; individual re-identification
across production cycles; feed-formulation optimisation; and the near-field and far-field
thermal physics (the thermal notebook). Fin, skin and gill scoring would need close-range
imaging this rig does not have.

### 12.3 Repository checklist

- [ ] notebook runs top to bottom on a fresh runtime, every `[VET]` line PASS
- [ ] dataset name, version and **licence** recorded; measurement ground truth described
- [ ] **calibration target, its dimensions and the calibration date** committed beside the weights
- [ ] split by tank/session, never by frame
- [ ] `LW`, `SPECIES`, `K_BAND`, `ASPECT_BAND`, `OWI` weights replaced with species- and
      life-stage-specific values, each with a FishBase or literature citation
- [ ] biomass reported **with its interval**, never as a point estimate
- [ ] welfare index labelled **PARTIAL**, with the uncovered indicators listed
- [ ] `aquaculture_handoff.json` returned to the thermal notebook so the reuse ceiling closes the loop
- [ ] conditional finding completed and **reviewed by a vet and an aquatic scientist before it
      leaves the team**